# Anchored Predictive Synergy — reproducible analysis notebook

This is the cleaned release version of the analysis notebook accompanying the Anchored Predictive Synergy manuscript.

## Scope

The notebook contains:
1. the four-state anchored/extension estimation pipeline;
2. repeated cross-validation and subject-level out-of-fold predictions;
3. negative cross-entropy Anchored Predictive Synergy estimation;
4. multiplicity-adjusted summaries and sensitivity analyses;
5. final drugomics heterogeneity analyses used in the manuscript;
6. manuscript and supplementary figure generation.

Exploratory debugging cells, repeated drafts of figures, path-search cells, and superseded subgroup analyses have been removed.

## Data availability

The U-BIOPRED source data are not redistributed by this repository. Users must obtain the data under the applicable cohort governance/data-access arrangements and reproduce the expected directory structure locally.

## Reproducibility

Set the environment variables `APS_DATA_DIR` and `APS_RESULT_DIR` before running this notebook. The notebook deliberately does not contain a user-specific absolute path.

**Important:** this release notebook has been cleaned and syntax-checked, but it cannot be executed end-to-end here because the private U-BIOPRED data and local IntegrAO installation are not included.


In [ ]:
# Repository-level paths
# Set these in your shell before launching Jupyter, for example:
# export APS_DATA_DIR=/path/to/ubiopred/Dataset/G2/VersionTwo/data
# export APS_RESULT_DIR=/path/to/results/FINAL_S1_30x5

from pathlib import Path
import os

DATA_DIR = Path(os.environ.get("APS_DATA_DIR", "./data")).expanduser().resolve()
RESULT_DIR = Path(os.environ.get("APS_RESULT_DIR", "./results/FINAL_S1_30x5")).expanduser().resolve()

print("DATA_DIR  :", DATA_DIR)
print("RESULT_DIR:", RESULT_DIR)


# Anchored Predictive Synergy — Corrected Estimation Pipeline

This notebook replaces the combinatorial `itertools.combinations` sweep in
`exacerbation_Prediction_150_features.ipynb` with a pipeline that actually
produces what the manuscript's Methods section (§2.1–§2.8) needs to
populate Results: per-extension 2×2 factorial runs (baseline, anchor-only,
extension-only, anchor+extension), negative cross-entropy as the
performance metric, a single canonical fold-assignment table reused
everywhere, and a per-subject/per-fold record that the §2.6 paired
bootstrap and §2.7 power analysis can resample from.

**Before running, verify the remaining `TODO` blocks below** — they encode
assumptions I made without access to your `integrao` source, and getting
them wrong will silently produce the wrong numbers rather than an error:

1. ~~Anchor composition~~ — **resolved**: keep `clinical` and `genotype`
   as two separate modality graphs fed into IntegrAO together (not
   concatenated into one feature block). This is what the pipeline
   already implements — no code change needed.
2. ~~Eosinophil% / FeNO column names~~ — **resolved**. Confirmed against
   the real `clinical.txt`: `eosinophils_percent` and
   `no_standard_flow_rate` (the latter is FeNO despite the name — cross-
   checked against `FeNO.xls`, values match exactly for shared subjects).
   Both are complete (zero missing) across the 234-subject anchor
   population, so T2 status is computable for the whole anchor cohort
   (177 T2-high / 57 T2-low at the pre-specified thresholds).
3. ~~Probability extraction from IntegrAO~~ — **resolved**. Traced through
   your actual `integrater.py`: `inference_supervised()` computes softmax
   probabilities internally but discards them after `argmax`. Fixed below
   in `get_predicted_probabilities()`, which also fixes a related latent
   bug — `inference_supervised()` receives `id_list` (the model's own
   subject-ID ordering) but never uses it to align output, relying
   instead on an order-matching assumption that happens to hold under
   this pipeline's design but isn't guaranteed by the API. See the
   markdown cell just above that function for details.
4. ~~Number of CV repeats (R)~~ — **resolved via pilot, not a fixed
   guess**. Section 6.5 below runs a small stabilization pilot on the
   sparsest extension and sets `N_REPEATS` to the point where further
   repeats stop meaningfully reducing SE(Ŝ1), rather than importing a
   round number from unrelated literature.

**Also worth knowing before you run this:** extension coverage is very
uneven, confirmed against your real data (via the independent R
cross-check): `prot_serum` (n=41), `trans_sput` (n=52), `metagen_sput`
(n=62) are far sparser than the rest — Ŝ1 for those three will have
noticeably wider bootstrap CIs than denser extensions like `lipid_urine`
(n=231), `atopy_exposures` (n=234, full anchor coverage), `metab_urine`
(n=228), or `drug_urine` (n=227). **Coverage for the four newly-added
extensions (`ct_insp`, `ct_exp`, `cytokine_plasma`, `cytokine_sput`) is
not yet known** — run the R overlap script (or just let this notebook's
own population computation print `n=` per extension) before interpreting
any results from them, since sparse coverage could put them in the same
wide-CI category as `prot_serum`/`trans_sput`/`metagen_sput`.

**`atopy_exposures` is categorical**, unlike every other extension here
(15 yes/no/uncertain items). It's one-hot encoded rather than scaled --
see `CATEGORICAL_MODALITIES` / `one_hot_encode_fixed()` below -- because
"uncertain" reads as an epistemic response rather than a midpoint between
no and yes, and forcing an ordinal scale would assume a monotonic
relationship the data doesn't support. Its SNF fusion metric is Jaccard,
not Euclidean, to match the binary-encoded representation.

**Four new extensions added, not yet validated against real data:**
`ct_insp`/`ct_exp` (quantitative CT features) and
`cytokine_plasma`/`cytokine_sput` (cytokine panels). These are routed
through the same numeric pipeline as the other continuous extensions
(StandardScaler + Euclidean by default in `metric_map`), but unlike
`atopy_exposures` I haven't seen these files' actual structure --
verify they're subject-indexed numeric feature matrices in the same
`index_col=0, tab-delimited, latin1` format as the rest before trusting
the loading step, and reconsider the Euclidean default if their feature
distributions turn out to need transformation first.

In [ ]:
import itertools, os, sys, time, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch_geometric.transforms as T
import snf
from scipy.spatial.distance import cdist
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")

# Add the parent directory of "integrao" to the Python path (unchanged from original)
module_path = os.path.abspath(os.path.join('../'))
if module_path not in sys.path:
    sys.path.append(module_path)

from integrao.dataset import GraphDataset
from integrao.main import dist2
from integrao.integrater import integrao_integrater, integrao_predictor


## 1. Configuration — verify every value in this cell

In [ ]:
# --- Paths (updated to match your current data layout) ---
data_dir = str(DATA_DIR)

paths = {
    "prot_serum":   os.path.join(data_dir, "Adult.Baseline.Proteomics.SOMASCAN.SERUM.txt"),
    "metab_urine":  os.path.join(data_dir, "Adult.Baseline.Metabolomics.Urine.txt"),
    "trans_blood":  os.path.join(data_dir, "Adult.Baseline.Transcriptomics.Blood.txt"),
    "drug_urine":   os.path.join(data_dir, "Adult.Baseline.Drugomics.Karolinska.Drug.Levels.Urine.txt"),
    "genotype":     os.path.join(data_dir, "Adult.Baseline.Genotyping.txt"),
    "metagen_sput": os.path.join(data_dir, "Adult.Baseline.Metagenomics.SPUTUM.txt"),
    "lipid_urine":  os.path.join(data_dir, "Adult.Baseline.Lipidomics.Karolinska.Eicosanoid.Urine.txt"),
    "trans_sput":   os.path.join(data_dir, "Adult.Baseline.Transcriptomics.Sputum.txt"),
    "clinical":     os.path.join(data_dir, "clinical.txt"),
    "atopy_exposures": os.path.join(data_dir, "atopy_expoxures_and_triggers.txt"),
    "ct_insp":         os.path.join(data_dir, "Adult.Baseline.ComputedTomography.Inspiration.txt"),
    "ct_exp":          os.path.join(data_dir, "Adult.Baseline.ComputedTomography.Expiration.txt"),
    "cytokine_plasma": os.path.join(data_dir, "Adult.Baseline.Plasma.Cytokines.txt"),
    "cytokine_sput":   os.path.join(data_dir, "Adult.Baseline.Sputum.Cytokines.txt"),
}
truelabel_path = os.path.join(data_dir, "followup.txt")

# --- Anchor / extension composition (manuscript §2.1) ---
ANCHOR_MODALITIES = ["clinical", "genotype"]
EXTENSION_MODALITIES = [
    "prot_serum", "metab_urine", "trans_blood", "drug_urine",
    "metagen_sput", "lipid_urine", "trans_sput", "atopy_exposures",
    "ct_insp", "ct_exp", "cytokine_plasma", "cytokine_sput",
]

# atopy_exposures is categorical (yes/no/uncertain across 15 items), unlike every other
# modality here. It is one-hot encoded rather than ordinal-encoded in run_config below:
# "uncertain" reads as an epistemic response ("don't know if this is my trigger"), not a
# midpoint between no and yes, so ordinal coding would impose a monotonic assumption the
# data doesn't support -- and could wash out "uncertain" itself carrying independent
# predictive signal (e.g. poorer symptom awareness), which one-hot preserves.
CATEGORICAL_MODALITIES = {"atopy_exposures"}
CATEGORICAL_LEVELS = ["no", "yes", "uncertain"]  # fixed order -> stable one-hot columns

# Confirmed against the actual clinical.txt (2026-08-09): both columns present,
# zero missingness across the 234-subject anchor population (clinical ∩ genotype ∩ followup).
# Both must be dropped from the anchor feature set in every configuration, per §2.1 / §2.8
# -- not only when the T2 stratification is active.
EOS_PCT_COLUMN = "eosinophils_percent"
FENO_COLUMN = "no_standard_flow_rate"  # NB: this is FeNO (fractional exhaled Nitric Oxide,
                                        # standard flow rate protocol) despite the cryptic name --
                                        # cross-checked against FeNO.xls, values match exactly.
T2_EOS_THRESHOLD = 2.0                # blood eosinophil percentage >= 2%
T2_FENO_THRESHOLD = 25.0              # FeNO >= 25 ppb

# --- Distance metric per modality for SNF fusion (unchanged from original, plus atopy_exposures) ---
metric_map = {
    "metagen_sput": "braycurtis",
    "trans_blood": "euclidean", "trans_sput": "euclidean", "prot_serum": "euclidean",
    "lipid_urine": "euclidean", "metab_urine": "euclidean",
    "clinical": "euclidean", "genotype": "euclidean", "drug_urine": "euclidean",
    # One-hot encoded binary block -> Euclidean isn't meaningful here. Jaccard scores
    # overlap of *shared present* categories (shared "yes"/"uncertain" answers) without
    # inflating similarity from the many shared "no" answers the way Hamming would.
    "atopy_exposures": "jaccard",
    # Quantitative CT features and cytokine panels are continuous, same treatment as
    # proteomics/lipidomics/metabolomics above -- TODO: verify Euclidean is a reasonable
    # default once you've inspected these features' actual distributions (e.g. if CT
    # features are highly skewed or cytokine values span several orders of magnitude,
    # a log-transform before scaling, or a different distance, may fit better).
    "ct_insp": "euclidean", "ct_exp": "euclidean",
    "cytokine_plasma": "euclidean", "cytokine_sput": "euclidean",
}

# --- CV design (manuscript §2.3) ---
K_FOLDS = 5
N_REPEATS = 1          # TODO (#4): set to the agreed R before final runs
MAX_FEATURES = 150     # matches §2.3 and the notebook's original cell 9 (NOT cell 10's k=10)
RANDOM_SEED = 42

output_dir = "AnchoredSynergy_Run_" + time.strftime("%Y%m%d")
os.makedirs(output_dir, exist_ok=True)


## 2. Load data and build the label

In [ ]:
data_map = {name: pd.read_csv(p, index_col=0, delimiter="\t", encoding="latin1")
            for name, p in paths.items()}

truelabel = pd.read_csv(truelabel_path, index_col=0, delimiter="\t", encoding="latin1")
truelabel["target"] = (truelabel["cluster.id"] >= 2).astype(int)
print(truelabel["target"].value_counts().sort_index())


### Missingness check on the four newly-added extensions

`ct_insp`, `ct_exp`, `cytokine_plasma`, and `cytokine_sput` previously
failed every `(0,1)`/`(1,1)` fold because `StandardScaler` raises on any
NaN by default -- `run_config` now imputes (median, fit on the training
fold only) before scaling, but that can't help a feature that's 100%
missing within a given fold. Check for fully-empty columns before
trusting the fix silently worked.

In [ ]:
for ext in ["ct_insp", "ct_exp", "cytokine_plasma", "cytokine_sput"]:
    df = data_map[ext]
    pct_missing = df.isna().mean().sort_values(ascending=False)
    fully_empty = (pct_missing == 1.0).sum()
    print(f"{ext}: {df.shape[0]} subjects x {df.shape[1]} features, "
          f"{fully_empty} fully-empty columns, "
          f"median missingness per feature = {pct_missing.median():.1%}, "
          f"max missingness per feature = {pct_missing.max():.1%}")
    if fully_empty > 0:
        print(f"  WARNING: {fully_empty} column(s) are 100% missing across ALL subjects -- "
              f"these will still cause problems in small folds. Consider dropping them: "
              f"data_map['{ext}'] = data_map['{ext}'].drop(columns=pct_missing[pct_missing == 1.0].index)")


## 3. Anchor feature exclusion and T2 status (§2.1, §2.8)

Both the eosinophil% and FeNO columns are dropped from `clinical` here,
once, so every downstream configuration — pooled and T2-stratified —
uses an anchor with an identical definition, as §2.1 requires.

In [ ]:
clinical_raw = data_map["clinical"]

missing_cols = [c for c in (EOS_PCT_COLUMN, FENO_COLUMN) if c not in clinical_raw.columns]
if missing_cols:
    raise KeyError(
        f"{missing_cols} not found in clinical.txt columns. "
        "Update EOS_PCT_COLUMN / FENO_COLUMN in the config cell to the actual header names "
        "before proceeding -- silently skipping this check would let eos%/FeNO leak into the anchor."
    )

# T2 status computed BEFORE the columns are dropped from the anchor
t2_status = (
    (clinical_raw[FENO_COLUMN] >= T2_FENO_THRESHOLD) |
    (clinical_raw[EOS_PCT_COLUMN] >= T2_EOS_THRESHOLD)
).astype(int)
t2_status.name = "t2_high"
print(t2_status.value_counts())

# Now drop them from the anchor feature set, uniformly, per §2.1/§2.8
data_map["clinical"] = clinical_raw.drop(columns=[EOS_PCT_COLUMN, FENO_COLUMN])


## 4. Fusion helper (unchanged logic from the original notebook)

In [ ]:
def get_custom_fused_network(obj, datasets, names, ids):
    """Inject custom per-modality distance metrics into Integrater/Predictor fusion.
    Unchanged from the original notebook's cell 7."""
    S_dfs = []
    module = sys.modules[obj.__module__]
    integrao_fuse = module.integrao_fuse
    for name, df in zip(names, datasets):
        metric = metric_map.get(name, "euclidean")
        d_mat = cdist(df.values, df.values, metric=metric)
        S_mat = snf.compute.affinity_matrix(d_mat, K=obj.neighbor_size, mu=obj.mu)
        S_dfs.append(pd.DataFrame(S_mat, index=ids, columns=ids))
    return integrao_fuse(S_dfs, obj.dicts_common, obj.dicts_unique, obj.original_order,
                         obj.neighbor_size, obj.fusing_iteration, obj.normalization_factor)


## 5. Fold-assignment table builder (§2.3)

Defines `build_fold_table` only. It's invoked twice: once by the pilot in
Section 6.5 below (to choose `N_REPEATS`), and once afterward to build the
actual canonical table used for the main run — so the canonical table can
use whatever `N_REPEATS` the pilot settles on, rather than a value guessed
in advance.

In [ ]:
def build_fold_table(subject_ids, y, t2, k=K_FOLDS, n_repeats=N_REPEATS, seed=RANDOM_SEED):
    """Returns a long-format DataFrame: subject_id, repeat, fold.
    Attempts joint stratification by outcome x T2 status; falls back to
    outcome-only stratification if any joint stratum is too small for k folds."""
    subject_ids = list(subject_ids)
    y = y.loc[subject_ids]
    t2_aligned = t2.reindex(subject_ids)

    joint = (y.astype(str) + "_" + t2_aligned.astype(str))
    joint_counts = joint.value_counts()
    strata = joint if joint_counts.min() >= k else y
    if joint_counts.min() < k:
        print(f"  [fold table] joint outcome x T2 stratification infeasible "
              f"(smallest stratum={joint_counts.min()} < k={k}); falling back to outcome-only.")

    records = []
    for r in range(n_repeats):
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed + r)
        for fold_idx, (_, test_idx) in enumerate(skf.split(subject_ids, strata)):
            for i in test_idx:
                records.append({"subject_id": subject_ids[i], "repeat": r, "fold": fold_idx})
    return pd.DataFrame.from_records(records)


## 6. Per-configuration training/scoring

`run_config` handles all four (i, j) cells of the 2x2 design:

- **(0, 0) baseline** — no modalities at all. Per §2.2, this has a
  closed form: predict the training-fold empirical prevalence p̄ for
  every test subject and score cross-entropy against that constant.
  No model is trained.
- **(1, 0) anchor-only**, **(0, 1) extension-only**, **(1, 1) both** —
  routed through IntegrAO exactly as the original notebook did for any
  modality subset, just restricted to the specific anchor/extension
  combination for a given (i, j) rather than an arbitrary combinatorial
  sweep.

**On `unsupervised_alignment()` — removed, confirmed wasted compute.**
Real run logs showed every trained config printing two full training
phases: `integrater.unsupervised_alignment()` (up to 1000 epochs, no
`clf_loss`) followed immediately by `classification_finetuning()`, which
prints *"Pre-trained weights not found. Training from scratch"* and
retrains from a random initialization regardless. Traced through
`integrater.py`: `classification_finetuning()` only reads
`self.fused_networks` and the indexing dicts from `__init__` — nothing
`unsupervised_alignment()` sets. And `run_config` never used its return
value. So that first phase was fully discarded work, and in the logs it
was often the *larger* of the two costs per fit, not the smaller one.
`run_config` below no longer calls it.

### On probability extraction (§2.2) — resolved against your actual `integrao/integrater.py`

`integrao_predictor.inference_supervised()` already computes class
probabilities internally — `F.softmax(preds, dim=1)` — but then calls
`np.argmax(...)` on the very next line and returns only the hard label,
discarding the probabilities. Getting −CE just requires stopping one step
earlier.

There's a second, more important issue in that same function:
`inference_supervised()` also receives `id_list` back from the model's
`forward()` (`IntegrAO_supervised.py`) — the subject-ID order the model
actually produced its predictions in — but never uses it. It just returns
`preds` and implicitly trusts that the order matches the input order.
Tracing through `forward()`: `z_sample_dict` is built by iterating domains
and inserting each `sample_id` into a Python dict the first time it's
seen, so the output order matches input order *as long as every modality
in a config shares an identical subject set* — true throughout this
pipeline's design, so it happens to be safe here. But it's an incidental
property of dict insertion order under this specific design, not something
the API guarantees, and a silent misalignment here would corrupt every
downstream θ̂ and Ŝ1 with no error raised. `get_predicted_probabilities()`
below captures `id_list` and explicitly reindexes to `te_ids`, removing
that risk rather than relying on it continuing to hold.

In [ ]:
def get_predicted_probabilities(predictor, model_path, te_list, combo_names):
    """Reimplements integrao_predictor.inference_supervised() up to the
    softmax step (see integrater.py lines 384-419), but returns class
    probabilities instead of the argmax'd hard label, and explicitly aligns
    output to subject IDs via the model's own id_list rather than assuming
    output order matches input order (see markdown cell above for why that
    assumption, while currently safe under this pipeline's design, isn't
    something the underlying API guarantees)."""
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    from integrao.IntegrAO_supervised import IntegrAO
    model = IntegrAO(
        predictor.feature_dims, predictor.hidden_channels, predictor.embedding_dims,
        num_classes=predictor.num_classes,
    ).to(device)
    model = predictor._load_pre_trained_weights(model, model_path, device)
    model.eval()

    x_dict, edge_index_dict = {}, {}
    for i, modal in enumerate(te_list):
        model_name = combo_names[i]
        modal_index = predictor.modalities_name_list.index(model_name)
        dataset = GraphDataset(
            predictor.neighbor_size, modal.values,
            predictor.fused_networks[modal_index].values,
            transform=T.ToDevice(device),
        )
        modal_dg = dataset[0]
        x_dict[modal_index] = modal_dg.x
        edge_index_dict[modal_index] = modal_dg.edge_index

    with torch.no_grad():
        _, _, logits, id_list = model(x_dict, edge_index_dict, predictor.dict_original_order)
        probs = F.softmax(logits, dim=1).detach().cpu().numpy()

    # class 1 = positive label (target == 1, i.e. >=2 exacerbations), matching
    # how `truelabel["target"]` is constructed in the data-loading cell above.
    prob_series = pd.Series(probs[:, 1], index=id_list)

    p_hat = prob_series.reindex(te_list[0].index).values
    if np.isnan(p_hat).any():
        missing = prob_series.index.symmetric_difference(te_list[0].index)
        raise ValueError(
            f"get_predicted_probabilities: {np.isnan(p_hat).sum()} test subjects have no "
            f"matching prediction after aligning by id_list. Mismatched subjects: "
            f"{list(missing)[:10]}. This means the incidental order-matching assumption "
            f"noted above has broken for this config -- do not silently proceed."
        )
    return p_hat


def neg_cross_entropy(y_true, p_hat, eps=1e-12):
    p = np.clip(p_hat, eps, 1 - eps)
    return -(-(y_true * np.log(p) + (1 - y_true) * np.log(1 - p)))  # -CE, higher = better


def one_hot_encode_fixed(df, levels=CATEGORICAL_LEVELS):
    """One-hot encode every column against a FIXED set of levels, so train and test
    folds always produce the same columns even if a level happens not to appear in one
    of them (e.g. nobody answered 'uncertain' to a given item in a small test fold)."""
    out = pd.DataFrame(index=df.index)
    for col in df.columns:
        for lvl in levels:
            out[f"{col}__{lvl}"] = (df[col] == lvl).astype(int)
    return out


def run_config(modality_names, tr_ids, te_ids, y, tmp_root, finetune_epochs=200):
    """Returns a DataFrame: subject_id, y_true, p_hat, neg_ce for the test subjects,
    for one (i, j) configuration on one fold."""
    y_tr = y.loc[tr_ids]
    y_te = y.loc[te_ids]

    if len(modality_names) == 0:
        # (0, 0) baseline -- closed-form null, no model trained (§2.2)
        p_bar = y_tr.mean()
        p_hat = np.full(len(te_ids), p_bar)
        return pd.DataFrame({
            "subject_id": te_ids, "y_true": y_te.values, "p_hat": p_hat,
            "neg_ce": neg_cross_entropy(y_te.values, p_hat),
        })

    tr_list, te_list = [], []
    for name in modality_names:
        if name in CATEGORICAL_MODALITIES:
            # One-hot, not scaled: Jaccard (see metric_map) expects binary-like input,
            # and StandardScaler on 0/1 dummies would break that assumption.
            tr_sc = one_hot_encode_fixed(data_map[name].loc[tr_ids])
            te_sc = one_hot_encode_fixed(data_map[name].loc[te_ids])
            tr_sc, te_sc = tr_sc.values, te_sc.values
        else:
            # Impute BEFORE scaling, fit on the training fold only (never on test),
            # so no information leaks across the split. Median is a defensible
            # default for arbitrary omics/imaging features without per-modality
            # domain knowledge about the right imputation strategy -- ct_insp,
            # ct_exp, cytokine_plasma, and cytokine_sput all failed every (0,1)/
            # (1,1) fold before this was added, because StandardScaler raises on
            # any NaN by default and these four files (unlike the other 8) carry
            # real missingness in their feature columns.
            imputer = SimpleImputer(strategy="median")
            tr_imp = imputer.fit_transform(data_map[name].loc[tr_ids])
            te_imp = imputer.transform(data_map[name].loc[te_ids])

            scaler = StandardScaler()
            tr_sc = scaler.fit_transform(tr_imp)
            te_sc = scaler.transform(te_imp)

        k_sel = min(MAX_FEATURES, tr_sc.shape[1])
        selector = SelectKBest(score_func=f_classif, k=k_sel)
        tr_final = selector.fit_transform(tr_sc, y_tr)
        te_final = selector.transform(te_sc)

        tr_list.append(pd.DataFrame(tr_final, index=tr_ids))
        te_list.append(pd.DataFrame(te_final, index=te_ids))

    integrater = integrao_integrater(tr_list, modalities_name_list=modality_names)
    integrater.fused_networks = get_custom_fused_network(integrater, tr_list, modality_names, tr_ids)

    tmp_dir = os.path.join(tmp_root, "tmp")
    _, _, model, _ = integrater.classification_finetuning(y_tr, tmp_dir, finetune_epochs=finetune_epochs)

    predictor = integrao_predictor(te_list, modalities_name_list=modality_names, num_classes=2)
    predictor.fused_networks = get_custom_fused_network(predictor, te_list, modality_names, te_ids)

    temp_pth = os.path.join(tmp_root, "fold.pth")
    torch.save(model.state_dict(), temp_pth)
    p_hat = np.asarray(get_predicted_probabilities(predictor, temp_pth, te_list, modality_names), dtype=float)

    return pd.DataFrame({
        "subject_id": te_ids, "y_true": y_te.values, "p_hat": p_hat,
        "neg_ce": neg_cross_entropy(y_te.values, p_hat),
    })


## 6.5. Pilot: choosing N_REPEATS empirically

Rather than fixing R to a round number, this runs a small pilot to find
where additional repeats stop meaningfully reducing SE(Ŝ1), and sets
`N_REPEATS` from that.

**What R actually reduces.** The §2.6 bootstrap (5,000 subject-level
resamples) estimates *sampling variance* — uncertainty from which subjects
happened to be in the cohort. R is a separate variance source: for a fixed
set of subjects, which ones land in the train vs. test fold on any given
5-fold split is itself arbitrary, and a single split is one noisy draw
from that arbitrariness (*partition variance*). Repeating CV and averaging
reduces partition variance; it does nothing to sampling variance, which is
why both the bootstrap and R are needed and are not substitutes for each
other.

**Why a pilot rather than a literature default.** Repeated-CV variance
reduction consistently shows diminishing returns after the first several
repeats, but exactly where the plateau falls depends on the dataset —
there's no unbiased estimator of CV variance in general (Bengio &
Grandvalet, 2004), which is itself an argument for checking empirically on
this cohort rather than importing a number from an unrelated benchmark.

**Design of the pilot.** Run on `trans_sput`, the sparsest extension
(n=65) — the worst case for partition variance, and the case where
getting R wrong costs the most. For R_MAX_PILOT repeats, compute Ŝ1 once
per repeat, then look at how the standard error of the running mean of
those per-repeat Ŝ1 values shrinks as more repeats are included —
averaged over `N_PERM` random orderings of the repeats, so the curve
isn't an artifact of the particular sequence they happened to run in.
`N_REPEATS` is set to the smallest R within 10% of the SE achieved at
R_MAX_PILOT.

**Cost.** This trains `R_MAX_PILOT x K_FOLDS x 3` models (300 at the
defaults below — the (0,0) baseline is closed-form and free, so only 3 of
the 4 configs actually train). Each trained fit runs
`classification_finetuning()` (`finetune_epochs`, default 200) —
`unsupervised_alignment()` is no longer called (see the note in Section 6
above: it trained a model that `classification_finetuning()` never used,
so it was pure wasted compute and has been removed from `run_config`).
The cell below times one fit on your machine before committing to all
300, and prints an estimated total runtime so you can adjust
`R_MAX_PILOT` / `PILOT_FINETUNE_EPOCHS` first if needed, rather than
finding out after launching the full loop.

In [ ]:
import matplotlib.pyplot as plt

PILOT_EXTENSION = "trans_sput"   # sparsest extension (n=65) -- worst case for partition variance
R_MAX_PILOT = 20                  # upper bound to search over
N_PERM = 30                       # random re-orderings of the R_MAX_PILOT repeats, averaged for
                                   # a smoother curve -- post-hoc only, no extra model fits
PILOT_FINETUNE_EPOCHS = 200       # lower this to speed up the pilot; see cost note above

pilot_population = sorted(
    set(truelabel.index) & set(data_map["clinical"].index)
    & set(data_map["genotype"].index) & set(data_map[PILOT_EXTENSION].index)
)
print(f"Pilot extension: {PILOT_EXTENSION} (n={len(pilot_population)})")

pilot_fold_table = build_fold_table(
    pilot_population, truelabel["target"], t2_status, n_repeats=R_MAX_PILOT
)

pilot_configs = {
    (0, 0): [],
    (1, 0): list(ANCHOR_MODALITIES),
    (0, 1): [PILOT_EXTENSION],
    (1, 1): list(ANCHOR_MODALITIES) + [PILOT_EXTENSION],
}
y_pilot = truelabel.loc[pilot_population, "target"]

# --- Timing probe: run ONE trained config once, before committing to the full pilot ---
# 3 trained configs per (repeat, fold) -- (0,0) is closed-form and effectively free.
n_trained_fits = R_MAX_PILOT * K_FOLDS * 3

_probe_fold_idx = sorted(pilot_fold_table["fold"].unique())[0]
_probe_te = pilot_fold_table.loc[
    (pilot_fold_table["repeat"] == 0) & (pilot_fold_table["fold"] == _probe_fold_idx), "subject_id"
].tolist()
_probe_tr = [s for s in pilot_population if s not in set(_probe_te)]
_probe_tmp = os.path.join(output_dir, "tmp_models_pilot_probe")
os.makedirs(_probe_tmp, exist_ok=True)

_t0 = time.time()
_ = run_config(list(ANCHOR_MODALITIES), _probe_tr, _probe_te, y_pilot, _probe_tmp,
               finetune_epochs=PILOT_FINETUNE_EPOCHS)
_probe_seconds = time.time() - _t0

_est_total = _probe_seconds * n_trained_fits
print(f"\nOne fit took {_probe_seconds:.1f}s on this machine.")
print(f"Pilot needs {n_trained_fits} trained fits (R_MAX_PILOT={R_MAX_PILOT} x K_FOLDS={K_FOLDS} x 3 trained configs).")
print(f"Estimated pilot runtime: ~{_est_total/60:.1f} minutes (~{_est_total/3600:.1f} hours) "
      f"if every fit takes about the same time.")
print("If that's too slow: lower R_MAX_PILOT, or lower PILOT_FINETUNE_EPOCHS above. "
      "Re-run this cell after changing them to get an updated estimate before "
      "committing to the full loop below.")
print("Caveat: the probe used the 2-modality anchor-only config. The (1,1) config fuses "
      "3 modality graphs and will cost somewhat more per fit, so this estimate is a "
      "plausible lower bound, not an exact figure.")

pilot_records = []
for r in range(R_MAX_PILOT):
    for fold_idx in sorted(pilot_fold_table["fold"].unique()):
        te_ids = pilot_fold_table.loc[
            (pilot_fold_table["repeat"] == r) & (pilot_fold_table["fold"] == fold_idx), "subject_id"
        ].tolist()
        tr_ids = [s for s in pilot_population if s not in set(te_ids)]
        if not te_ids or not tr_ids:
            print(f"  [skip] pilot repeat={r} fold={fold_idx}: empty train or test set")
            continue

        tmp_root = os.path.join(output_dir, "tmp_models_pilot", f"r{r}_f{fold_idx}")
        os.makedirs(tmp_root, exist_ok=True)

        for (i, j), modality_names in pilot_configs.items():
            try:
                df = run_config(modality_names, tr_ids, te_ids, y_pilot, tmp_root,
                                 finetune_epochs=PILOT_FINETUNE_EPOCHS)
            except Exception as e:
                print(f"  [FAILED] pilot config=({i},{j}) repeat={r} fold={fold_idx}: {e}")
                continue
            df["i"], df["j"], df["repeat"], df["fold"] = i, j, r, fold_idx
            pilot_records.append(df)

pilot_perf = pd.concat(pilot_records, ignore_index=True)
pilot_perf.to_csv(os.path.join(output_dir, "pilot_per_subject_performance.csv"), index=False)


def theta_hat_repeat(df, i, j, r):
    sub = df[(df["i"] == i) & (df["j"] == j) & (df["repeat"] == r)]
    return sub.groupby("fold")["neg_ce"].mean().mean()


s1_per_repeat = np.array([
    (theta_hat_repeat(pilot_perf, 1, 1, r) - theta_hat_repeat(pilot_perf, 1, 0, r))
    - (theta_hat_repeat(pilot_perf, 0, 1, r) - theta_hat_repeat(pilot_perf, 0, 0, r))
    for r in range(R_MAX_PILOT)
])

rng = np.random.default_rng(RANDOM_SEED)
se_curves = np.full((N_PERM, R_MAX_PILOT), np.nan)
for p in range(N_PERM):
    shuffled = s1_per_repeat[rng.permutation(R_MAX_PILOT)]
    for rp in range(2, R_MAX_PILOT + 1):  # SE undefined for a single repeat
        se_curves[p, rp - 1] = shuffled[:rp].std(ddof=1) / np.sqrt(rp)

mean_se_curve = np.nanmean(se_curves, axis=0)
r_values = np.arange(1, R_MAX_PILOT + 1)
asymptotic_se = mean_se_curve[-1]

plateau_mask = mean_se_curve <= asymptotic_se * 1.1
plateau_R = int(r_values[np.argmax(plateau_mask)]) if plateau_mask.any() else R_MAX_PILOT

print(f"\nSE(S1_hat) by number of repeats (averaged over {N_PERM} random orderings):")
for rp, se in zip(r_values, mean_se_curve):
    flag = "  <-- chosen" if rp == plateau_R else ""
    se_str = f"{se:.5f}" if not np.isnan(se) else "  n/a"
    print(f"  R={rp:2d}: SE={se_str}{flag}")

plt.figure(figsize=(6, 4))
plt.plot(r_values, mean_se_curve, marker="o")
plt.axvline(plateau_R, color="red", linestyle="--", label=f"chosen R={plateau_R}")
plt.xlabel("Number of CV repeats (R)")
plt.ylabel("SE(Ŝ1) across repeats")
plt.title(f"Repeated-CV stabilization pilot ({PILOT_EXTENSION}, n={len(pilot_population)})")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "N_REPEATS_pilot_diagnostic.png"), dpi=150)
plt.show()

N_REPEATS = plateau_R
print(f"\nN_REPEATS set to {N_REPEATS} based on this pilot.")
print(
    f'Manuscript sentence: "R was set to {N_REPEATS}, the smallest number of repeats within '
    f'10% of the SE(S1_hat) achieved at R={R_MAX_PILOT} in a pilot run on the sparsest '
    f'extension ({PILOT_EXTENSION}, n={len(pilot_population)}; Supplementary Fig. X)."'
)


## 6.6. Canonical fold-assignment table (§2.3)

Built now that `N_REPEATS` is settled by the pilot above, on the
population that has the anchor and the outcome (near-universal per §2.1),
jointly stratified by outcome and T2 status where per-fold counts allow
it. Every extension's analysis restricts this same table to its own
analysis population rather than re-splitting.

In [ ]:
anchor_population = sorted(
    set(truelabel.index) & set(data_map["clinical"].index) & set(data_map["genotype"].index)
)
canonical_fold_table = build_fold_table(anchor_population, truelabel["target"], t2_status,
                                         n_repeats=N_REPEATS)
canonical_fold_table.to_csv(os.path.join(output_dir, "canonical_fold_table.csv"), index=False)
print(f"Canonical fold table built on {len(anchor_population)} subjects "
      f"({N_REPEATS} repeat(s) x {K_FOLDS} folds).")


## 7. Main loop — one extension at a time, full 2x2, per-subject records

For each extension: the analysis population is subjects with the anchor,
outcome, *and that extension* measured (§2.1) — not a population shared
across extensions. The fold labels for that population are the
**restriction** of the single canonical table (§2.3), not a fresh split.

In [ ]:
CONFIGS = {
    (0, 0): [],
    (1, 0): ANCHOR_MODALITIES,
    (0, 1): None,   # filled in per-extension below
    (1, 1): None,   # filled in per-extension below
}

all_records = []

for ext in EXTENSION_MODALITIES:
    print(f"=== Extension: {ext} ===")
    population = sorted(
        set(truelabel.index)
        & set(data_map["clinical"].index) & set(data_map["genotype"].index)
        & set(data_map[ext].index)
    )
    n_pop = len(population)
    print(f"  analysis population n={n_pop}")

    fold_table_ext = canonical_fold_table[canonical_fold_table["subject_id"].isin(population)]
    # Subjects in the canonical table but missing this extension are dropped;
    # subjects with this extension but absent from the canonical table
    # (i.e. missing the anchor) cannot occur, since the canonical table's
    # base population already requires the anchor.

    configs_ext = {
        (0, 0): [],
        (1, 0): list(ANCHOR_MODALITIES),
        (0, 1): [ext],
        (1, 1): list(ANCHOR_MODALITIES) + [ext],
    }

    y_pop = truelabel.loc[population, "target"]

    for r in range(N_REPEATS):
        for fold_idx in sorted(fold_table_ext["fold"].unique()):
            te_ids = fold_table_ext.loc[
                (fold_table_ext["repeat"] == r) & (fold_table_ext["fold"] == fold_idx), "subject_id"
            ].tolist()
            tr_ids = [s for s in population if s not in set(te_ids)]

            if len(te_ids) == 0 or len(tr_ids) == 0:
                print(f"  [skip] repeat={r} fold={fold_idx}: empty train or test set after restriction")
                continue

            tmp_root = os.path.join(output_dir, "tmp_models", ext, f"r{r}_f{fold_idx}")
            os.makedirs(tmp_root, exist_ok=True)

            for (i, j), modality_names in configs_ext.items():
                try:
                    df = run_config(modality_names, tr_ids, te_ids, y_pop, tmp_root)
                except Exception as e:
                    print(f"  [FAILED] ext={ext} config=({i},{j}) repeat={r} fold={fold_idx}: {e}")
                    continue
                df["extension"] = ext
                df["i"] = i
                df["j"] = j
                df["repeat"] = r
                df["fold"] = fold_idx
                all_records.append(df)

per_subject_performance = pd.concat(all_records, ignore_index=True)
per_subject_performance.to_csv(os.path.join(output_dir, "per_subject_performance.csv"), index=False)
print(f"Wrote {len(per_subject_performance)} per-subject records to "
      f"{os.path.join(output_dir, 'per_subject_performance.csv')}")


## 8. Aggregate to θ̂ᵢⱼ and Ŝ1 per extension (§2.3–§2.5)

This just implements the estimator as written: mean over repeats of the
mean over folds. The bootstrap (§2.6) and power analysis (§2.7) should be
run against `per_subject_performance.csv` directly, not against this
aggregated table -- that file is what preserves the subject-level,
fold-aligned structure the bootstrap needs.

In [ ]:
def theta_hat(df, i, j):
    sub = df[(df["i"] == i) & (df["j"] == j)]
    fold_means = sub.groupby(["repeat", "fold"])["neg_ce"].mean()
    return fold_means.groupby("repeat").mean().mean()


rows = []
for ext in EXTENSION_MODALITIES:
    df_ext = per_subject_performance[per_subject_performance["extension"] == ext]
    if df_ext.empty:
        continue
    th00 = theta_hat(df_ext, 0, 0)
    th10 = theta_hat(df_ext, 1, 0)
    th01 = theta_hat(df_ext, 0, 1)
    th11 = theta_hat(df_ext, 1, 1)
    delta_hat = th11 - th10
    s1_hat = (th11 - th10) - (th01 - th00)
    rows.append({
        "extension": ext, "theta_00": th00, "theta_10": th10,
        "theta_01": th01, "theta_11": th11, "delta_hat": delta_hat, "S1_hat": s1_hat,
    })

summary = pd.DataFrame(rows).sort_values("S1_hat", ascending=False)
summary.to_csv(os.path.join(output_dir, "S1_summary.csv"), index=False)
summary


In [ ]:

import pandas as pd

RESULT_DIR = str(RESULT_DIR)

bootstrap_results = pd.read_csv(
    f"{RESULT_DIR}/S1_subject_bootstrap_5000.csv"
)


# Order all results by significance
ordered_results = (
    bootstrap_results
    .sort_values(
        ["p-value", "Metric", "Observed S1"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

display(ordered_results.round(4))

significant_results = (
    bootstrap_results[
        (bootstrap_results["95% CI lower"] > 0) |
        (bootstrap_results["95% CI upper"] < 0)
    ]
    .sort_values("p-value")
    .reset_index(drop=True)
)

display(significant_results.round(4))

In [ ]:
primary_results = (
    bootstrap_results[
        bootstrap_results["Metric"] == "Negative CE"
    ]
    .sort_values("FDR q-value")
    .reset_index(drop=True)
)

display(
    primary_results[
        [
            "Extension",
            "Observed S1",
            "95% CI lower",
            "95% CI upper",
            "p-value",
            "FDR q-value",
            "FDR significant",
            "Interpretation"
        ]
    ].round(4)
)

In [ ]:
secondary = (
    bootstrap_results[
        bootstrap_results["Metric"].isin(
            ["PR-AUC", "Weighted F1"]
        )
    ]
    .sort_values(
        ["Metric", "Observed S1"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(
    secondary[
        [
            "Extension",
            "Metric",
            "Observed S1",
            "95% CI lower",
            "95% CI upper",
            "p-value",
            "P(S1 > 0)",
            "Interpretation"
        ]
    ].round(4)
)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# Benjamini-Hochberg FDR
# ============================================================

def benjamini_hochberg(pvals, alpha=0.05):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)

    order = np.argsort(pvals)
    ranked = pvals[order]

    q_ranked = ranked * n / np.arange(1, n + 1)
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0, 1)

    qvals = np.empty(n)
    qvals[order] = q_ranked

    reject = qvals < alpha

    return qvals, reject


# ============================================================
# FDR separately within each metric family
# ============================================================

final_inference = bootstrap_results.copy()

final_inference["FDR q-value within metric"] = np.nan
final_inference["FDR significant within metric"] = False

for metric in ["Negative CE", "PR-AUC", "Weighted F1"]:

    mask = final_inference["Metric"] == metric

    qvals, reject = benjamini_hochberg(
        final_inference.loc[mask, "p-value"].values,
        alpha=0.05
    )

    final_inference.loc[
        mask, "FDR q-value within metric"
    ] = qvals

    final_inference.loc[
        mask, "FDR significant within metric"
    ] = reject


# ============================================================
# Clean ordered table
# ============================================================

final_table = (
    final_inference
    .sort_values(
        ["Metric", "FDR q-value within metric", "Observed S1"]
    )
    [
        [
            "Extension",
            "Metric",
            "Observed S1",
            "95% CI lower",
            "95% CI upper",
            "p-value",
            "FDR q-value within metric",
            "FDR significant within metric"
        ]
    ]
    .reset_index(drop=True)
)

display(final_table.round(4))

In [ ]:
# ============================================================
# FIGURE 1 — CLEAN MINIMAL VERSION
# No overlapping titles or annotations
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ------------------------------------------------------------
# Display order
# ------------------------------------------------------------

extensions_order = [
    "atopy",
    "drug",
    "metab",
    "protein",
    "lipid"
]

display_names = {
    "atopy": "Atopy / exposures",
    "drug": "Drugomics",
    "metab": "Metabolomics",
    "protein": "Proteomics",
    "lipid": "Lipidomics"
}

# ------------------------------------------------------------
# Restrained palette
# ------------------------------------------------------------

BLUE   = "#2166AC"
ORANGE = "#D95F02"
GREY   = "#777777"
BLACK  = "#222222"

# ------------------------------------------------------------
# Reorder
# ------------------------------------------------------------

factorial_plot = (
    factorial
    .set_index("extension")
    .loc[extensions_order]
    .reset_index()
)

s1_plot = (
    s1_pr
    .set_index("extension")
    .loc[extensions_order]
    .reset_index()
)

y = np.arange(len(extensions_order))


# ============================================================
# FIGURE
# ============================================================

fig, (ax1, ax2) = plt.subplots(
    1,
    2,
    figsize=(11.2, 4.8),
    gridspec_kw={
        "width_ratios": [1.05, 1.0],
        "wspace": 0.40
    }
)

# No suptitle.
# Give panels room.
fig.subplots_adjust(
    left=0.14,
    right=0.97,
    bottom=0.18,
    top=0.88
)


# ============================================================
# PANEL A — PREDICTIVE CONTRIBUTION
# ============================================================

offset = 0.12

for i, row in factorial_plot.iterrows():

    # --------------------------------------------------------
    # Extension alone
    # B00 -> B01
    # --------------------------------------------------------

    ax1.plot(
        [row["B00"], row["B01"]],
        [y[i] - offset, y[i] - offset],
        color=BLUE,
        lw=1.6
    )

    # Reference
    ax1.scatter(
        row["B00"],
        y[i] - offset,
        s=44,
        facecolor="white",
        edgecolor=BLUE,
        linewidth=1.5,
        zorder=3
    )

    # Result
    ax1.scatter(
        row["B01"],
        y[i] - offset,
        s=44,
        facecolor=BLUE,
        edgecolor=BLUE,
        zorder=3
    )


    # --------------------------------------------------------
    # Added to anchor
    # B10 -> B11
    # --------------------------------------------------------

    ax1.plot(
        [row["B10"], row["B11"]],
        [y[i] + offset, y[i] + offset],
        color=ORANGE,
        lw=1.6
    )

    # Anchor reference
    ax1.scatter(
        row["B10"],
        y[i] + offset,
        s=44,
        marker="s",
        facecolor="white",
        edgecolor=ORANGE,
        linewidth=1.5,
        zorder=3
    )

    # Anchor + extension
    ax1.scatter(
        row["B11"],
        y[i] + offset,
        s=44,
        marker="s",
        facecolor=ORANGE,
        edgecolor=ORANGE,
        zorder=3
    )


# ------------------------------------------------------------
# Subtle common reference lines
# NO text above them
# ------------------------------------------------------------

ax1.axvline(
    B00_pr,
    color=BLUE,
    lw=0.9,
    ls=":",
    alpha=0.65
)

ax1.axvline(
    B10_pr,
    color=ORANGE,
    lw=0.9,
    ls=":",
    alpha=0.65
)


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

ax1.set_yticks(y)

ax1.set_yticklabels(
    [display_names[e] for e in extensions_order],
    fontsize=9.5
)

ax1.invert_yaxis()

ax1.set_xlim(
    0.43,
    0.70
)

ax1.set_xlabel(
    "PR-AUC",
    fontsize=10
)

ax1.set_title(
    "A   Predictive contribution",
    loc="left",
    fontsize=11.5,
    fontweight="bold",
    pad=10
)


# ------------------------------------------------------------
# Compact legend underneath data
# ------------------------------------------------------------

legend_handles = [

    Line2D(
        [0], [0],
        color=BLUE,
        marker="o",
        markerfacecolor=BLUE,
        markeredgecolor=BLUE,
        markersize=5.5,
        lw=1.5,
        label="Extension alone"
    ),

    Line2D(
        [0], [0],
        color=ORANGE,
        marker="s",
        markerfacecolor=ORANGE,
        markeredgecolor=ORANGE,
        markersize=5.5,
        lw=1.5,
        label="Added to anchor"
    )
]

ax1.legend(
    handles=legend_handles,
    frameon=False,
    fontsize=8.2,
    loc="lower right",
    handlelength=2.2
)


# ============================================================
# PANEL B — S1 FOREST
# ============================================================

x = s1_plot["Observed S1"].values
lo = s1_plot["95% CI lower"].values
hi = s1_plot["95% CI upper"].values

xerr = np.vstack([
    x - lo,
    hi - x
])

ax2.errorbar(
    x,
    y,
    xerr=xerr,
    fmt="o",
    color=BLUE,
    ecolor=BLUE,
    markersize=5.2,
    lw=1.4,
    capsize=3
)

# Zero interaction
ax2.axvline(
    0,
    color=GREY,
    ls="--",
    lw=1.0
)


# ------------------------------------------------------------
# Q values in dedicated right-hand space
# ------------------------------------------------------------

for i, row in s1_plot.iterrows():

    q = row["q"]

    if q < 0.05:
        txt = f"q={q:.3f}"
        weight = "bold"
    else:
        txt = "ns"
        weight = "normal"

    ax2.text(
        0.058,
        y[i],
        txt,
        ha="left",
        va="center",
        fontsize=8.3,
        fontweight=weight
    )


# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

ax2.set_yticks(y)

ax2.set_yticklabels(
    [display_names[e] for e in extensions_order],
    fontsize=9.5
)

ax2.invert_yaxis()

# Extra room after zero purely for q-values
ax2.set_xlim(
    -0.27,
    0.115
)

ax2.set_xlabel(
    r"$S_1$ (PR-AUC)",
    fontsize=10
)

ax2.set_title(
    "B   Anchor–extension interaction",
    loc="left",
    fontsize=11.5,
    fontweight="bold",
    pad=10
)


# ============================================================
# MINIMAL AXES
# ============================================================

for ax in [ax1, ax2]:

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        axis="both",
        labelsize=9
    )


# ============================================================
# SMALL EXPLANATORY FOOTNOTES
# ============================================================

fig.text(
    0.14,
    0.055,
    "Open symbols denote the reference model; filled symbols denote performance after adding the extension.",
    fontsize=7.8,
    color=GREY
)

fig.text(
    0.66,
    0.055,
    r"$S_1<0$: sub-additive;  $S_1>0$: synergistic. "
    r"q values are BH-FDR adjusted within the PR-AUC family.",
    fontsize=7.8,
    color=GREY,
    ha="center"
)


# ============================================================
# SAVE
# ============================================================

pdf_path = os.path.join(
    FIG_DIR,
    "Figure1_UBIOPRED_minimal.pdf"
)

png_path = os.path.join(
    FIG_DIR,
    "Figure1_UBIOPRED_minimal.png"
)

tiff_path = os.path.join(
    FIG_DIR,
    "Figure1_UBIOPRED_minimal.tiff"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    tiff_path,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# FIGURE 2 — STANDALONE VS CONDITIONAL PREDICTIVE GAIN
# U-BIOPRED
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

extensions_order = [
    "atopy",
    "drug",
    "metab",
    "protein",
    "lipid"
]

display_names = {
    "atopy": "Atopy / exposures",
    "drug": "Drugomics",
    "metab": "Metabolomics",
    "protein": "Proteomics",
    "lipid": "Lipidomics"
}

# ------------------------------------------------------------
# Build gains from existing factorial table
# ------------------------------------------------------------

gain_df = (
    factorial
    .set_index("extension")
    .loc[extensions_order]
    .reset_index()
)

gain_df["Standalone gain"] = (
    gain_df["B01"] - gain_df["B00"]
)

gain_df["Conditional gain"] = (
    gain_df["B11"] - gain_df["B10"]
)

gain_df["S1"] = (
    gain_df["Conditional gain"]
    -
    gain_df["Standalone gain"]
)

display(
    gain_df[
        [
            "extension",
            "Standalone gain",
            "Conditional gain",
            "S1"
        ]
    ].round(4)
)

# ============================================================
# PLOT
# ============================================================

fig, ax = plt.subplots(
    figsize=(6.4, 5.6)
)

x = gain_df["Standalone gain"].values
y = gain_df["Conditional gain"].values

# ------------------------------------------------------------
# Reference lines
# ------------------------------------------------------------

ax.axhline(
    0,
    color="0.65",
    linewidth=1
)

ax.axvline(
    0,
    color="0.65",
    linewidth=1
)

# Additive line: conditional gain = standalone gain
lims = [
    min(x.min(), y.min()) - 0.015,
    max(x.max(), y.max()) + 0.015
]

ax.plot(
    lims,
    lims,
    linestyle="--",
    linewidth=1.1,
    color="0.35"
)

# ------------------------------------------------------------
# Points
# ------------------------------------------------------------

ax.scatter(
    x,
    y,
    s=65,
    zorder=3
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

label_offsets = {
    "atopy": (5, 6),
    "drug": (6, 6),
    "metab": (6, -14),
    "protein": (6, -14),
    "lipid": (6, 6)
}

for _, row in gain_df.iterrows():

    dx, dy = label_offsets[
        row["extension"]
    ]

    ax.annotate(
        display_names[
            row["extension"]
        ],
        (
            row["Standalone gain"],
            row["Conditional gain"]
        ),
        xytext=(dx, dy),
        textcoords="offset points",
        fontsize=9
    )

# ------------------------------------------------------------
# Axis labels
# ------------------------------------------------------------

ax.set_xlabel(
    r"Standalone gain: $B_{01}-B_{00}$ (PR-AUC)",
    fontsize=10
)

ax.set_ylabel(
    r"Conditional gain: $B_{11}-B_{10}$ (PR-AUC)",
    fontsize=10
)

ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_aspect(
    "equal",
    adjustable="box"
)

# ------------------------------------------------------------
# Minimal explanatory annotations
# ------------------------------------------------------------

ax.text(
    0.04,
    0.95,
    "Above diagonal: super-additive",
    transform=ax.transAxes,
    fontsize=8.5,
    va="top"
)

ax.text(
    0.04,
    0.90,
    "Below diagonal: sub-additive",
    transform=ax.transAxes,
    fontsize=8.5,
    va="top"
)

# ------------------------------------------------------------
# Clean style
# ------------------------------------------------------------

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(
    axis="both",
    labelsize=9
)

ax.set_title(
    "Standalone predictive value does not imply incremental value",
    fontsize=11.5,
    fontweight="bold",
    pad=10
)

plt.tight_layout()

# ============================================================
# SAVE
# ============================================================

pdf_path = os.path.join(
    FIG_DIR,
    "Figure2_UBIOPRED_standalone_vs_conditional.pdf"
)

png_path = os.path.join(
    FIG_DIR,
    "Figure2_UBIOPRED_standalone_vs_conditional.png"
)

tiff_path = os.path.join(
    FIG_DIR,
    "Figure2_UBIOPRED_standalone_vs_conditional.tiff"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    tiff_path,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(pdf_path)
print(png_path)
print(tiff_path)

In [ ]:
# ============================================================
# SUPPLEMENTARY FIGURE S1
# Robustness of S1 across predictive metrics
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

extensions_order = [
    "atopy",
    "drug",
    "metab",
    "protein",
    "lipid"
]

display_names = {
    "atopy": "Atopy / exposures",
    "drug": "Drugomics",
    "metab": "Metabolomics",
    "protein": "Proteomics",
    "lipid": "Lipidomics"
}

metrics = [
    "Negative CE",
    "PR-AUC",
    "Weighted F1"
]

panel_titles = {
    "Negative CE": "A   Negative CE",
    "PR-AUC": "B   PR-AUC",
    "Weighted F1": "C   Weighted F1"
}

# ------------------------------------------------------------
# Prepare table
# ------------------------------------------------------------

plot_df = final_inference.copy()

plot_df["extension_key"] = (
    plot_df["Extension"]
    .str.lower()
)

plot_df["extension_key"] = pd.Categorical(
    plot_df["extension_key"],
    categories=extensions_order,
    ordered=True
)

plot_df = plot_df.sort_values(
    ["Metric", "extension_key"]
)

# ============================================================
# FIGURE
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(12.5, 4.7),
    sharey=True
)

fig.subplots_adjust(
    left=0.15,
    right=0.98,
    bottom=0.18,
    top=0.88,
    wspace=0.28
)

y = np.arange(len(extensions_order))

# ============================================================
# DRAW PANELS
# ============================================================

for ax, metric in zip(axes, metrics):

    d = (
        plot_df[
            plot_df["Metric"] == metric
        ]
        .set_index("extension_key")
        .loc[extensions_order]
        .reset_index()
    )

    x = d["Observed S1"].values
    lo = d["95% CI lower"].values
    hi = d["95% CI upper"].values

    xerr = np.vstack([
        x - lo,
        hi - x
    ])

    # -----------------------------------------
    # Confidence intervals + point estimates
    # -----------------------------------------

    ax.errorbar(
        x,
        y,
        xerr=xerr,
        fmt="o",
        markersize=5.5,
        linewidth=1.4,
        capsize=3
    )

    # -----------------------------------------
    # Null interaction
    # -----------------------------------------

    ax.axvline(
        0,
        color="0.45",
        linestyle="--",
        linewidth=1
    )

    # -----------------------------------------
    # Titles
    # -----------------------------------------

    ax.set_title(
        panel_titles[metric],
        loc="left",
        fontsize=11,
        fontweight="bold",
        pad=10
    )

    ax.set_xlabel(
        r"$S_1$",
        fontsize=10
    )

    # -----------------------------------------
    # Clean axes
    # -----------------------------------------

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        axis="both",
        labelsize=9
    )

    # -----------------------------------------
    # FDR significance
    # -----------------------------------------

    for i, row in d.iterrows():

        q = row["FDR q-value within metric"]

        if q < 0.05:
            label = "*"
            weight = "bold"
        else:
            label = ""

        if label:

            # place star slightly beyond CI
            span = hi.max() - lo.min()
            offset = span * 0.035

            ax.text(
                hi[i] + offset,
                y[i],
                label,
                fontsize=11,
                fontweight=weight,
                va="center"
            )


# ============================================================
# Y AXIS LABELS
# ============================================================

axes[0].set_yticks(y)

axes[0].set_yticklabels(
    [
        display_names[e]
        for e in extensions_order
    ],
    fontsize=9.5
)

axes[0].invert_yaxis()

# Make labels visible despite sharey
for ax in axes[1:]:
    ax.tick_params(
        axis="y",
        left=False,
        labelleft=False
    )


# ============================================================
# ADD SIMPLE DIRECTION LABELS AT BOTTOM
# ============================================================

fig.text(
    0.50,
    0.06,
    r"$S_1 < 0$: sub-additive       $S_1 = 0$: additive       $S_1 > 0$: synergistic",
    ha="center",
    fontsize=8.5,
    color="0.35"
)


# ============================================================
# SAVE
# ============================================================

pdf_path = os.path.join(
    FIG_DIR,
    "FigureS1_UBIOPRED_S1_metric_robustness.pdf"
)

png_path = os.path.join(
    FIG_DIR,
    "FigureS1_UBIOPRED_S1_metric_robustness.png"
)

tiff_path = os.path.join(
    FIG_DIR,
    "FigureS1_UBIOPRED_S1_metric_robustness.tiff"
)

fig.savefig(
    pdf_path,
    bbox_inches="tight"
)

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    tiff_path,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(pdf_path)
print(png_path)
print(tiff_path)

In [ ]:
# ============================================================
# U-BIOPRED S1 SENSITIVITY / MINIMUM DETECTABLE EFFECT ANALYSIS
# Primary metric: Negative cross-entropy
#
# Purpose:
# Quantify the approximate S1 interaction magnitude required
# for 80% power at two-sided alpha = 0.05.
#
# This is a sensitivity/MDE analysis, not "observed power".
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

BASE = str(RESULT_DIR)

FILE = os.path.join(
    BASE,
    "S1_subject_bootstrap_5000.csv"
)

df = pd.read_csv(FILE)

# ------------------------------------------------------------
# PRIMARY METRIC
# ------------------------------------------------------------

ce = df[
    df["Metric"]
    .astype(str)
    .str.strip()
    .str.lower()
    == "negative ce"
].copy()

# ------------------------------------------------------------
# Constants
#
# two-sided alpha=.05:
# z_(1-alpha/2) = 1.959964
#
# 80% power:
# z_.80 = 0.841621
#
# sum ≈ 2.8016
# ------------------------------------------------------------

Z_ALPHA = 1.959964
Z_POWER = 0.841621

MDE_MULTIPLIER = (
    Z_ALPHA + Z_POWER
)

# ------------------------------------------------------------
# Use bootstrap SD already estimated from 5000 resamples
# ------------------------------------------------------------

ce["SE_bootstrap"] = pd.to_numeric(
    ce["Bootstrap SD"],
    errors="coerce"
)

ce["MDE_80_abs_S1"] = (
    MDE_MULTIPLIER
    *
    ce["SE_bootstrap"]
)

# Express observed effect relative to detection threshold
ce["Observed_abs_S1"] = (
    pd.to_numeric(
        ce["Observed S1"],
        errors="coerce"
    )
    .abs()
)

ce["Observed_as_fraction_of_MDE"] = (
    ce["Observed_abs_S1"]
    /
    ce["MDE_80_abs_S1"]
)

# ------------------------------------------------------------
# Also estimate detectable S1 interval around zero
# ------------------------------------------------------------

ce["Detectable_positive_S1"] = (
    ce["MDE_80_abs_S1"]
)

ce["Detectable_negative_S1"] = (
    -ce["MDE_80_abs_S1"]
)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

out = ce[
    [
        "Extension",
        "Observed S1",
        "Bootstrap SD",
        "95% CI lower",
        "95% CI upper",
        "p-value",
        "FDR q-value",
        "MDE_80_abs_S1",
        "Observed_as_fraction_of_MDE"
    ]
].copy()

out = out.rename(
    columns={
        "Observed S1":
            "Observed_S1",

        "Bootstrap SD":
            "Bootstrap_SE",

        "95% CI lower":
            "CI_lower",

        "95% CI upper":
            "CI_upper",

        "p-value":
            "p_value",

        "FDR q-value":
            "FDR_q"
    }
)

out = out.sort_values(
    "MDE_80_abs_S1"
).reset_index(drop=True)

print("=" * 100)
print("S1 MINIMUM DETECTABLE EFFECT ANALYSIS")
print("Primary metric: Negative cross-entropy")
print("Two-sided alpha = 0.05; target power = 80%")
print(
    "MDE multiplier =",
    round(MDE_MULTIPLIER, 4),
    "× bootstrap SE"
)
print("=" * 100)

display(
    out.round(4)
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

OUT_FILE = os.path.join(
    BASE,
    "S1_negative_CE_minimum_detectable_effect_80pct.csv"
)

out.to_csv(
    OUT_FILE,
    index=False
)

print("\nSaved:")
print(OUT_FILE)

In [ ]:
# =============================================================================
# ANALYSIS NAME:
# ANALYSIS_DRUGOMICS_CLUSTER1_VS_CLUSTER0_126_VS_89_FULL_CLINICAL
#
# PURPOSE:
# Full clinical characterization of the two ACTUAL drugomics clusters.
#
# Cluster 1 = 126
# Cluster 0 = 89
# Total clustered severe-asthma population = 215
#
# Uses:
#   1) authoritative cluster membership file from the previous analysis
#   2) full U-BIOPRED clinical dataset
#
# =============================================================================

import os
import numpy as np
import pandas as pd

from scipy.stats import (
    mannwhitneyu,
    fisher_exact,
    chi2_contingency
)

from statsmodels.stats.multitest import multipletests


# =============================================================================
# 1. PATHS
# =============================================================================

CLINICAL_FILE = (
    "/Users/nz1413/IntegrAO/data/ubiopred/"
    "UBIOPRED_FOr_MAnuscript_Nazanin_For_Joe_V4_MedicationAdded.xlsx"
)

BASE = (
    "/Users/nz1413/IntegrAO/results/ubiopred/"
    "FINAL_S1_30x5/subgroup_S1/"
    "drugomics_feature_dissection"
)

CLUSTER_FILE = os.path.join(
    BASE,
    "FINAL_CLUSTER1_VS_CLUSTER0_126_VS_89",
    "SUBJECTS_drugomics_cluster1_vs_cluster0_126_vs_89.csv"
)

OUT_DIR = os.path.join(
    BASE,
    "FINAL_CLUSTER1_VS_CLUSTER0_126_VS_89_FULL_CLINICAL"
)

os.makedirs(
    OUT_DIR,
    exist_ok=True
)

print("=" * 110)
print("ANALYSIS_DRUGOMICS_CLUSTER1_VS_CLUSTER0_126_VS_89_FULL_CLINICAL")
print("=" * 110)


# =============================================================================
# 2. LOAD DATA
# =============================================================================

clusters = pd.read_csv(
    CLUSTER_FILE
)

clinical = pd.read_excel(
    CLINICAL_FILE
)

print("\nCluster file shape:")
print(clusters.shape)

print("\nClinical file shape:")
print(clinical.shape)


# =============================================================================
# 3. IDENTIFY SUBJECT ID COLUMNS
# =============================================================================

def find_id_col(df):

    candidates = [
        "patient",
        "Patient",
        "subject",
        "Subject",
        "subject_id",
        "Subject_ID",
        "Patient_ID",
        "ID",
        "id"
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(
        "Could not identify subject ID column."
    )


CLUSTER_ID = find_id_col(
    clusters
)

CLINICAL_ID = find_id_col(
    clinical
)

print("\nCluster ID column:", CLUSTER_ID)
print("Clinical ID column:", CLINICAL_ID)


# =============================================================================
# 4. STANDARDIZE IDS BEFORE MERGE
# =============================================================================

clusters[CLUSTER_ID] = (
    clusters[CLUSTER_ID]
    .astype(str)
    .str.strip()
)

clinical[CLINICAL_ID] = (
    clinical[CLINICAL_ID]
    .astype(str)
    .str.strip()
)


# =============================================================================
# 5. HARD CHECK CLUSTER MEMBERSHIP
# =============================================================================

clusters["Cluster1"] = pd.to_numeric(
    clusters["Cluster1"],
    errors="coerce"
)

clusters = clusters[
    clusters["Cluster1"].isin([0, 1])
].copy()

print("\nOriginal cluster counts:")
print(
    clusters["Cluster1"]
    .value_counts()
    .sort_index()
)

if len(clusters) != 215:
    raise ValueError(
        f"Expected 215 cluster subjects; found {len(clusters)}."
    )

if int((clusters["Cluster1"] == 1).sum()) != 126:
    raise ValueError(
        "Expected Cluster 1 n=126."
    )

if int((clusters["Cluster1"] == 0).sum()) != 89:
    raise ValueError(
        "Expected Cluster 0 n=89."
    )


# =============================================================================
# 6. MERGE CLUSTER LABELS INTO FULL CLINICAL DATA
# =============================================================================

cluster_small = clusters[
    [
        CLUSTER_ID,
        "Cluster1"
    ]
].copy()

cluster_small = cluster_small.rename(
    columns={
        CLUSTER_ID: "merge_id"
    }
)

clinical = clinical.rename(
    columns={
        CLINICAL_ID: "merge_id"
    }
)

merged = cluster_small.merge(
    clinical,
    on="merge_id",
    how="left",
    validate="one_to_one"
)

print("\nMerged shape:")
print(merged.shape)

print("\nCluster counts after merge:")
print(
    merged["Cluster1"]
    .value_counts()
    .sort_index()
)

missing_full_rows = (
    merged
    .drop(columns=["merge_id", "Cluster1"])
    .isna()
    .all(axis=1)
    .sum()
)

print(
    "\nSubjects with no clinical row matched:",
    missing_full_rows
)

if missing_full_rows > 0:
    raise ValueError(
        "Some cluster subjects did not match the clinical dataset."
    )

if len(merged) != 215:
    raise ValueError(
        f"Merge changed N. Expected 215, found {len(merged)}."
    )

print("\n✓ Correct full-clinical population confirmed: 126 vs 89")


# =============================================================================
# 7. SHOW RELEVANT AVAILABLE VARIABLES
# =============================================================================

keywords = [
    "Age",
    "Sex",
    "Gender",
    "Body",
    "BMI",
    "Smoking",
    "FEV1",
    "FVC",
    "AQLQ",
    "ACQ",
    "Exacerb",
    "Sputum",
    "Neut",
    "Eosin",
    "FeNO",
    "Corticosteroid",
    "OCS",
    "IgE",
    "Atopy",
    "MARS",
    "AsthmaBurgen",
    "MD_per_year",
    "sumED",
    "ICU",
    "hosp"
]

relevant_cols = []

for c in merged.columns:
    if any(
        k.lower() in c.lower()
        for k in keywords
    ):
        relevant_cols.append(c)

print("\n" + "=" * 110)
print("RELEVANT CLINICAL COLUMNS FOUND")
print("=" * 110)

for c in relevant_cols:
    print(c)


# =============================================================================
# 8. DEFINE CONTINUOUS VARIABLES
# =============================================================================

continuous_candidates = [

    "Age",
    "Body_Mass_Index_kgm2",

    "FEV1_Pre_Salbutamol",
    "FEV1_Post_Salbutamol",
    "FEV1FVC_Post_Salbutamol_Actual_Ratio",
    "FEV1_Change",
    "FVC_Pre_Salbutamol",

    "sputum_Neutrophils",
    "sputum_Eosinophils",

    "Exacerbation_Per_Year",

    "AQLQ_Average",
    "ACQ5_Total_Raw",

    "AsthmaBurgenSCore",
    "AsthmaBurgenSCore_weighted",

    "MARS_Total_Raw",

    "MD_per_year",
    "sumED_per_year",
    "hosp_LOS",
    "ICU_LOS",

    "Oral_Corticosteroids_Normalised_Dose_.mg."
]

continuous_vars = [
    c for c in continuous_candidates
    if c in merged.columns
]

print("\nContinuous variables detected:")
print(continuous_vars)


# =============================================================================
# 9. MISSINGNESS TABLE
# =============================================================================

missingness_rows = []

for var in continuous_vars:

    for cluster_value, group_name in [
        (1, "Cluster1"),
        (0, "Cluster0")
    ]:

        x = pd.to_numeric(
            merged.loc[
                merged["Cluster1"] == cluster_value,
                var
            ],
            errors="coerce"
        )

        missingness_rows.append(
            {
                "Variable": var,
                "Group": group_name,
                "N_total": len(x),
                "N_available": x.notna().sum(),
                "N_missing": x.isna().sum(),
                "Percent_missing": 100 * x.isna().mean()
            }
        )

missingness = pd.DataFrame(
    missingness_rows
)

display(
    missingness.round(2)
)


# =============================================================================
# 10. CONTINUOUS CHARACTERISTICS
# =============================================================================

continuous_results = []

for var in continuous_vars:

    x1 = pd.to_numeric(
        merged.loc[
            merged["Cluster1"] == 1,
            var
        ],
        errors="coerce"
    ).dropna()

    x0 = pd.to_numeric(
        merged.loc[
            merged["Cluster1"] == 0,
            var
        ],
        errors="coerce"
    ).dropna()

    if len(x1) < 5 or len(x0) < 5:
        continue

    u, p = mannwhitneyu(
        x1,
        x0,
        alternative="two-sided"
    )

    rank_biserial = (
        2 * u
        /
        (len(x1) * len(x0))
        - 1
    )

    q1_1, med1, q3_1 = np.percentile(
        x1,
        [25, 50, 75]
    )

    q1_0, med0, q3_0 = np.percentile(
        x0,
        [25, 50, 75]
    )

    continuous_results.append(
        {
            "Variable": var,

            "Cluster1_N": len(x1),
            "Cluster1_median_IQR":
                f"{med1:.2f} [{q1_1:.2f}, {q3_1:.2f}]",

            "Cluster0_N": len(x0),
            "Cluster0_median_IQR":
                f"{med0:.2f} [{q1_0:.2f}, {q3_0:.2f}]",

            "Rank_biserial": rank_biserial,
            "Mann_Whitney_U": u,
            "p_value": p
        }
    )


continuous_results = pd.DataFrame(
    continuous_results
)

if len(continuous_results) > 0:

    continuous_results[
        "FDR_q"
    ] = multipletests(
        continuous_results["p_value"],
        method="fdr_bh"
    )[1]

    continuous_results[
        "FDR_significant"
    ] = (
        continuous_results["FDR_q"]
        < 0.05
    )

    continuous_results = (
        continuous_results
        .sort_values(
            ["FDR_q", "p_value"]
        )
        .reset_index(drop=True)
    )


print("\n")
print("=" * 110)
print("FULL CONTINUOUS CLINICAL CHARACTERISTICS")
print("Drugomics Cluster 1 (126) vs Cluster 0 (89)")
print("=" * 110)

display(
    continuous_results.round(4)
)


# =============================================================================
# 11. CATEGORICAL VARIABLES
# =============================================================================

categorical_candidates = [

    "Sex",
    "Gender",
    "Smoking_Status",

    "Oral_Corticosteroids",
    "Inhaled_Corticosteroids",
    "Injectable_Corticosteroids",

    "Anti_IgE_Therapy",

    "IgE_Assay_Atopy_Result",
    "Skin_Prick_Test_Atopy_Result",
    "Combined_Atopy_Result_Common_Aeroallergens",
    "Combined_Atopy_Result_Food_Allergens",
    "Combined_Atopy_Result_All_Allergens",

    "Diabetes_Diagnosed",
    "Gerd_Diagnosed",
    "Hypertension_Diagnosed",
    "Sinusitis_Diagnosed",

    "Race"
]

categorical_vars = [
    c for c in categorical_candidates
    if c in merged.columns
]

categorical_results = []

categorical_tables = {}

for var in categorical_vars:

    temp = merged[
        [
            "Cluster1",
            var
        ]
    ].dropna()

    if len(temp) == 0:
        continue

    table = pd.crosstab(
        temp["Cluster1"],
        temp[var]
    )

    categorical_tables[var] = table

    if table.shape[0] != 2:
        continue

    if table.shape == (2, 2):

        odds_ratio, p = fisher_exact(
            table.values
        )

        test = "Fisher exact"
        statistic = odds_ratio

    else:

        chi2, p, dof, expected = (
            chi2_contingency(
                table.values
            )
        )

        test = "Chi-square"
        statistic = chi2

    categorical_results.append(
        {
            "Variable": var,
            "Test": test,
            "Statistic": statistic,
            "p_value": p
        }
    )


categorical_results = pd.DataFrame(
    categorical_results
)

if len(categorical_results) > 0:

    categorical_results[
        "FDR_q"
    ] = multipletests(
        categorical_results["p_value"],
        method="fdr_bh"
    )[1]

    categorical_results[
        "FDR_significant"
    ] = (
        categorical_results["FDR_q"]
        < 0.05
    )

    categorical_results = (
        categorical_results
        .sort_values(
            ["FDR_q", "p_value"]
        )
        .reset_index(drop=True)
    )


print("\n")
print("=" * 110)
print("FULL CATEGORICAL CLINICAL CHARACTERISTICS")
print("=" * 110)

display(
    categorical_results.round(4)
)


# =============================================================================
# 12. PRINT CATEGORICAL CONTINGENCY TABLES
# =============================================================================

print("\n")
print("=" * 110)
print("CATEGORICAL CONTINGENCY TABLES")
print("=" * 110)

for var, table in categorical_tables.items():

    print("\n---", var, "---")
    display(table)


# =============================================================================
# 13. SAVE EVERYTHING
# =============================================================================

merged.to_csv(
    os.path.join(
        OUT_DIR,
        "DATASET_cluster1_vs_cluster0_126_vs_89_full_clinical.csv"
    ),
    index=False
)

continuous_results.to_csv(
    os.path.join(
        OUT_DIR,
        "TABLE_continuous_full_clinical_cluster1_vs_cluster0.csv"
    ),
    index=False
)

categorical_results.to_csv(
    os.path.join(
        OUT_DIR,
        "TABLE_categorical_full_clinical_cluster1_vs_cluster0.csv"
    ),
    index=False
)

missingness.to_csv(
    os.path.join(
        OUT_DIR,
        "TABLE_missingness_full_clinical_cluster1_vs_cluster0.csv"
    ),
    index=False
)


# =============================================================================
# 14. SAVE ALL CATEGORICAL TABLES INTO ONE LONG CSV
# =============================================================================

cat_long = []

for var, table in categorical_tables.items():

    temp = (
        table
        .reset_index()
        .melt(
            id_vars="Cluster1",
            var_name="Category",
            value_name="N"
        )
    )

    temp["Variable"] = var

    cat_long.append(
        temp
    )

if len(cat_long) > 0:

    cat_long = pd.concat(
        cat_long,
        ignore_index=True
    )

    cat_long.to_csv(
        os.path.join(
            OUT_DIR,
            "TABLE_categorical_counts_full_clinical_cluster1_vs_cluster0.csv"
        ),
        index=False
    )


# =============================================================================
# 15. ANALYSIS MANIFEST
# =============================================================================

manifest = pd.DataFrame(
    {
        "Item": [
            "Analysis_name",
            "Clinical_file",
            "Cluster_membership_file",
            "Population",
            "Cluster1_N",
            "Cluster0_N",
            "Total_N",
            "Continuous_test",
            "Continuous_effect_size",
            "Categorical_test",
            "Multiple_testing",
            "Primary_comparator"
        ],

        "Value": [
            "ANALYSIS_DRUGOMICS_CLUSTER1_VS_CLUSTER0_126_VS_89_FULL_CLINICAL",
            CLINICAL_FILE,
            CLUSTER_FILE,
            "Drugomics-complete severe-asthma clustering population",
            126,
            89,
            215,
            "Two-sided Mann-Whitney U",
            "Rank-biserial correlation",
            "Fisher exact for 2x2; chi-square otherwise",
            "Benjamini-Hochberg FDR within continuous and categorical families",
            "Drugomics Cluster 1 versus Drugomics Cluster 0"
        ]
    }
)

manifest.to_csv(
    os.path.join(
        OUT_DIR,
        "README_ANALYSIS_MANIFEST_FULL_CLINICAL.csv"
    ),
    index=False
)


print("\n")
print("=" * 110)
print("ANALYSIS COMPLETE")
print("=" * 110)

print(
    "\nAnalysis name:\n"
    "ANALYSIS_DRUGOMICS_CLUSTER1_VS_CLUSTER0_126_VS_89_FULL_CLINICAL"
)

print(
    "\nResults saved to:\n",
    OUT_DIR
)

In [ ]:
# =============================================================================
# ANALYSIS NAME:
# ANALYSIS_DRUGOMICS_CLUSTER1_VS_CLUSTER0_126_VS_89_MULTIVARIABLE_BOOTSTRAP5000
#
# Outcome:
#   Cluster1 = 1 versus Cluster0 = 0
#
# Population:
#   Exact drugomics-complete clustered severe-asthma population
#   Cluster 1 = 126
#   Cluster 0 = 89
#   Total = 215
#
# Continuous predictors are standardized.
# Binary/dummy predictors are NOT standardized.
#
# Bootstrap:
#   5000 subject-level resamples
# =============================================================================

import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.multitest import multipletests


# =============================================================================
# 1. PATHS
# =============================================================================

BASE = (
    "/Users/nz1413/IntegrAO/results/ubiopred/"
    "FINAL_S1_30x5/subgroup_S1/"
    "drugomics_feature_dissection/"
    "FINAL_CLUSTER1_VS_CLUSTER0_126_VS_89_FULL_CLINICAL"
)

DATA_FILE = os.path.join(
    BASE,
    "DATASET_cluster1_vs_cluster0_126_vs_89_full_clinical.csv"
)

OUT_DIR = os.path.join(
    BASE,
    "MULTIVARIABLE_BOOTSTRAP5000"
)

os.makedirs(
    OUT_DIR,
    exist_ok=True
)


# =============================================================================
# 2. LOAD
# =============================================================================

d = pd.read_csv(
    DATA_FILE
)

print("=" * 100)
print(
    "ANALYSIS_DRUGOMICS_CLUSTER1_VS_CLUSTER0_"
    "126_VS_89_MULTIVARIABLE_BOOTSTRAP5000"
)
print("=" * 100)

print("\nDataset shape:", d.shape)

print("\nCluster counts:")
print(
    d["Cluster1"]
    .value_counts()
    .sort_index()
)


# =============================================================================
# 3. HARD POPULATION CHECK
# =============================================================================

if len(d) != 215:
    raise ValueError(
        f"Expected N=215; found {len(d)}."
    )

if int((d["Cluster1"] == 1).sum()) != 126:
    raise ValueError(
        "Expected Cluster1 n=126."
    )

if int((d["Cluster1"] == 0).sum()) != 89:
    raise ValueError(
        "Expected Cluster0 n=89."
    )

print("\n✓ Correct population confirmed: 126 vs 89")


# =============================================================================
# 4. SEX CODING
# =============================================================================

print("\nSex values:")
print(
    d["Sex"]
    .value_counts(dropna=False)
)

sex_string = (
    d["Sex"]
    .astype(str)
    .str.strip()
    .str.lower()
)

d["sex_female"] = (
    sex_string
    .isin([
        "female",
        "f",
        "woman",
        "women"
    ])
    .astype(int)
)


# =============================================================================
# 5. SMOKING DUMMIES
# =============================================================================

print("\nSmoking values:")
print(
    d["Smoking_Status"]
    .value_counts(dropna=False)
)

smoking = (
    d["Smoking_Status"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# reference category will effectively be the omitted smoking category
smoke_dummies = pd.get_dummies(
    smoking,
    prefix="smoking",
    drop_first=True,
    dtype=int
)

d = pd.concat(
    [
        d.reset_index(drop=True),
        smoke_dummies.reset_index(drop=True)
    ],
    axis=1
)

smoke_cols = list(
    smoke_dummies.columns
)

print("\nSmoking dummy variables:")
print(smoke_cols)


# =============================================================================
# 6. PRIMARY CLINICAL PREDICTORS
# =============================================================================

continuous = [

    "Age",
    "Body_Mass_Index_kgm2",

    "Exacerbation_Per_Year",

    "FEV1_Pre_Salbutamol",



    "AQLQ_Average",

    "ACQ5_Total_Raw"
]

binary = [
    "sex_female"
] + smoke_cols


# =============================================================================
# 7. CHECK AVAILABILITY
# =============================================================================

print("\nPredictor availability:")
for c in continuous + binary:
    print(
        f"{c:45s}",
        c in d.columns,
        (
            d[c].notna().sum()
            if c in d.columns
            else "NA"
        )
    )


# =============================================================================
# 8. PREP MODEL DATA
# =============================================================================

cols = (
    ["Cluster1"]
    + binary
    + continuous
)

model_data = d[
    cols
].copy()

for c in continuous:
    model_data[c] = pd.to_numeric(
        model_data[c],
        errors="coerce"
    )

model_data["Cluster1"] = pd.to_numeric(
    model_data["Cluster1"],
    errors="coerce"
)

model_data = (
    model_data
    .dropna()
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("COMPLETE-CASE MULTIVARIABLE POPULATION")
print("=" * 100)

print("N =", len(model_data))

print("\nCluster counts:")
print(
    model_data["Cluster1"]
    .value_counts()
    .sort_index()
)


# =============================================================================
# IMPORTANT:
# sputum neutrophils may reduce N substantially.
# We therefore report complete-case N explicitly.
# =============================================================================


# =============================================================================
# 9. DESIGN MATRIX FUNCTION
# =============================================================================

def make_design(
    df,
    scaler=None,
    fit_scaler=False
):

    # binary predictors remain 0/1
    X_binary = (
        df[binary]
        .astype(float)
        .values
    )

    # continuous predictors standardized
    X_cont = (
        df[continuous]
        .astype(float)
        .values
    )

    if fit_scaler:

        scaler = StandardScaler()

        X_cont = (
            scaler
            .fit_transform(X_cont)
        )

    else:

        X_cont = (
            scaler
            .transform(X_cont)
        )

    X = np.column_stack(
        [
            X_binary,
            X_cont
        ]
    )

    return X, scaler


# =============================================================================
# 10. OBSERVED MODEL
# =============================================================================

X, scaler = make_design(
    model_data,
    fit_scaler=True
)

y = (
    model_data["Cluster1"]
    .astype(int)
    .values
)

predictor_names = (
    binary
    + continuous
)

model = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=5000,
    random_state=42
)

model.fit(
    X,
    y
)

observed_beta = (
    model
    .coef_[0]
)


# =============================================================================
# 11. SUBJECT BOOTSTRAP
# =============================================================================

N_BOOT = 5000

rng = np.random.default_rng(
    42
)

n = len(
    model_data
)

boot_beta = np.full(
    (
        N_BOOT,
        len(predictor_names)
    ),
    np.nan
)

for b in range(N_BOOT):

    idx = rng.integers(
        0,
        n,
        n
    )

    db = (
        model_data
        .iloc[idx]
        .copy()
    )

    if (
        db["Cluster1"]
        .nunique()
        < 2
    ):
        continue

    try:

        Xb, _ = make_design(
            db,
            fit_scaler=True
        )

        yb = (
            db["Cluster1"]
            .astype(int)
            .values
        )

        mb = LogisticRegression(
            penalty="l2",
            C=1.0,
            max_iter=5000
        )

        mb.fit(
            Xb,
            yb
        )

        boot_beta[b, :] = (
            mb.coef_[0]
        )

    except Exception:
        continue


# =============================================================================
# 12. BOOTSTRAP SUMMARY
# =============================================================================

rows = []

for j, predictor in enumerate(
    predictor_names
):

    values = boot_beta[:, j]

    values = values[
        np.isfinite(values)
    ]

    obs = observed_beta[j]

    lo = np.quantile(
        values,
        0.025
    )

    hi = np.quantile(
        values,
        0.975
    )

    p = min(
        1.0,
        2 * min(
            np.mean(values <= 0),
            np.mean(values >= 0)
        )
    )

    stability = (
        np.mean(values > 0)
        if obs > 0
        else
        np.mean(values < 0)
    )

    rows.append(
        {
            "Predictor": predictor,
            "Beta": obs,
            "OR": np.exp(obs),
            "CI_lower": np.exp(lo),
            "CI_upper": np.exp(hi),
            "p_value": p,
            "Directional_stability":
                stability,
            "N_model":
                len(model_data)
        }
    )


results = pd.DataFrame(
    rows
)


# =============================================================================
# 13. BH FDR
# =============================================================================

results["FDR_q"] = multipletests(
    results["p_value"],
    method="fdr_bh"
)[1]

results["FDR_significant"] = (
    results["FDR_q"] < 0.05
)

results = (
    results
    .sort_values(
        [
            "FDR_q",
            "p_value"
        ]
    )
    .reset_index(drop=True)
)


# =============================================================================
# 14. DISPLAY
# =============================================================================

print("\n")
print("=" * 110)
print(
    "MULTIVARIABLE PREDICTORS OF "
    "DRUGOMICS CLUSTER 1 MEMBERSHIP"
)
print(
    "Cluster 1 (126) versus Cluster 0 (89)"
)
print("=" * 110)

display(
    results.round(4)
)


# =============================================================================
# 15. SAVE RESULTS
# =============================================================================

RESULT_FILE = os.path.join(
    OUT_DIR,
    "TABLE_multivariable_cluster1_vs_cluster0_bootstrap5000.csv"
)

COEF_FILE = os.path.join(
    OUT_DIR,
    "BOOTSTRAP_coefficients_cluster1_vs_cluster0_5000.csv"
)

DATA_OUT = os.path.join(
    OUT_DIR,
    "DATASET_complete_case_multivariable_cluster1_vs_cluster0.csv"
)

results.to_csv(
    RESULT_FILE,
    index=False
)

pd.DataFrame(
    boot_beta,
    columns=predictor_names
).to_csv(
    COEF_FILE,
    index=False
)

model_data.to_csv(
    DATA_OUT,
    index=False
)


# =============================================================================
# 16. SAVE ANALYSIS MANIFEST
# =============================================================================

manifest = pd.DataFrame(
    {
        "Item": [
            "Analysis_name",
            "Population",
            "Outcome",
            "Original_N",
            "Complete_case_N",
            "Cluster1_original_N",
            "Cluster0_original_N",
            "Bootstrap_replicates",
            "Continuous_scaling",
            "Binary_scaling",
            "Model",
            "Multiple_testing",
            "Random_seed"
        ],

        "Value": [
            (
                "ANALYSIS_DRUGOMICS_CLUSTER1_VS_CLUSTER0_"
                "126_VS_89_MULTIVARIABLE_BOOTSTRAP5000"
            ),
            (
                "Drugomics-complete severe-asthma "
                "clustering population"
            ),
            "Cluster1 membership",
            215,
            len(model_data),
            126,
            89,
            5000,
            "StandardScaler",
            "None; retained as 0/1",
            "L2-penalized logistic regression",
            "Benjamini-Hochberg FDR",
            42
        ]
    }
)

manifest.to_csv(
    os.path.join(
        OUT_DIR,
        "README_ANALYSIS_MANIFEST_MULTIVARIABLE.csv"
    ),
    index=False
)


print("\n" + "=" * 100)
print("DONE")
print("=" * 100)

print("\nSaved:")
print(RESULT_FILE)
print(COEF_FILE)
print(DATA_OUT)

print("\nOutput directory:")
print(OUT_DIR)

In [ ]:
# =============================================================================
# FINAL METRIC DIAGNOSTIC
# ANALYSIS_SUBJECT_AGGREGATED_METRICS_AND_CALIBRATION
#
# All metrics are calculated from the SAME subject-averaged probabilities
# used by the primary negative-cross-entropy analysis.
# =============================================================================

import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    log_loss,
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)

from sklearn.linear_model import LogisticRegression


BASE = str(RESULT_DIR)

OUT_DIR = os.path.join(
    BASE,
    "CE_VS_PRAUC_CALIBRATION_DIAGNOSTICS"
)

os.makedirs(
    OUT_DIR,
    exist_ok=True
)

extensions = {
    "atopy": {
        "B00": "B00_OOF_predictions.csv",
        "B01": "B01_atopy_OOF_predictions.csv",
        "B10": "B10_anchor_OOF_predictions.csv",
        "B11": "B11_atopy_OOF_predictions.csv"
    },
    "drug": {
        "B00": "B00_OOF_predictions.csv",
        "B01": "B01_drug_OOF_predictions.csv",
        "B10": "B10_anchor_OOF_predictions.csv",
        "B11": "B11_drug_OOF_predictions.csv"
    },
    "lipid": {
        "B00": "B00_OOF_predictions.csv",
        "B01": "B01_lipid_OOF_predictions.csv",
        "B10": "B10_anchor_OOF_predictions.csv",
        "B11": "B11_lipid_OOF_predictions.csv"
    },
    "metab": {
        "B00": "B00_OOF_predictions.csv",
        "B01": "B01_metab_OOF_predictions.csv",
        "B10": "B10_anchor_OOF_predictions.csv",
        "B11": "B11_metab_OOF_predictions.csv"
    },
    "protein": {
        "B00": "B00_OOF_predictions.csv",
        "B01": "B01_protein_OOF_predictions.csv",
        "B10": "B10_anchor_OOF_predictions.csv",
        "B11": "B11_protein_OOF_predictions.csv"
    }
}

EPS = 1e-8


def load_subject_prob(filename):

    x = pd.read_csv(
        os.path.join(BASE, filename)
    )

    x["p1"] = pd.to_numeric(
        x["p1"],
        errors="coerce"
    )

    x["y_true"] = pd.to_numeric(
        x["y_true"],
        errors="coerce"
    )

    x = x.dropna(
        subset=["subject", "y_true", "p1"]
    )

    # THIS IS THE PRIMARY ANALYSIS AGGREGATION
    s = (
        x.groupby("subject")
        .agg(
            y_true=("y_true", "first"),
            p1=("p1", "mean")
        )
        .reset_index()
    )

    s["p1"] = np.clip(
        s["p1"],
        EPS,
        1-EPS
    )

    return s


def calibration(y, p):

    logit_p = np.log(
        p / (1-p)
    ).reshape(-1, 1)

    try:
        m = LogisticRegression(
            penalty=None,
            max_iter=5000
        )
        m.fit(logit_p, y)

    except Exception:
        m = LogisticRegression(
            penalty="l2",
            C=1e8,
            max_iter=5000
        )
        m.fit(logit_p, y)

    return (
        float(m.intercept_[0]),
        float(m.coef_[0][0])
    )


rows = []

for extension, files in extensions.items():

    for config, filename in files.items():

        x = load_subject_prob(
            filename
        )

        y = x["y_true"].astype(int).values
        p = x["p1"].values

        cal_int, cal_slope = calibration(
            y,
            p
        )

        rows.append({
            "Extension": extension,
            "Configuration": config,
            "N": len(x),

            "Negative_CE":
                -log_loss(
                    y,
                    p,
                    labels=[0,1]
                ),

            "PR_AUC":
                average_precision_score(
                    y,
                    p
                ),

            "ROC_AUC":
                roc_auc_score(
                    y,
                    p
                ),

            "Negative_Brier":
                -brier_score_loss(
                    y,
                    p
                ),

            "Calibration_intercept":
                cal_int,

            "Calibration_slope":
                cal_slope
        })


metrics = pd.DataFrame(rows)


# =============================================================================
# S1 FOR EACH METRIC
# =============================================================================

s1_rows = []

for extension in extensions:

    temp = (
        metrics[
            metrics["Extension"] == extension
        ]
        .set_index("Configuration")
    )

    for metric in [
        "Negative_CE",
        "PR_AUC",
        "ROC_AUC",
        "Negative_Brier"
    ]:

        B00 = temp.loc["B00", metric]
        B01 = temp.loc["B01", metric]
        B10 = temp.loc["B10", metric]
        B11 = temp.loc["B11", metric]

        standalone = B01 - B00
        conditional = B11 - B10

        s1_rows.append({
            "Extension": extension,
            "Metric": metric,
            "B00": B00,
            "B01": B01,
            "B10": B10,
            "B11": B11,
            "Standalone_gain": standalone,
            "Conditional_gain": conditional,
            "S1": conditional - standalone
        })


s1 = pd.DataFrame(
    s1_rows
)

wide = (
    s1.pivot(
        index="Extension",
        columns="Metric",
        values="S1"
    )
    .reset_index()
)


print("=" * 100)
print("SUBJECT-AGGREGATED CONFIGURATION METRICS")
print("=" * 100)

display(
    metrics.round(4)
)

print("\n")
print("=" * 100)
print("SUBJECT-AGGREGATED S1 BY METRIC")
print("=" * 100)

display(
    wide.round(4)
)


metrics.to_csv(
    os.path.join(
        OUT_DIR,
        "FINAL_subject_aggregated_configuration_metrics.csv"
    ),
    index=False
)

s1.to_csv(
    os.path.join(
        OUT_DIR,
        "FINAL_subject_aggregated_S1_by_metric.csv"
    ),
    index=False
)

wide.to_csv(
    os.path.join(
        OUT_DIR,
        "FINAL_subject_aggregated_S1_wide.csv"
    ),
    index=False
)

print("\nSaved to:")
print(OUT_DIR)

In [ ]:
# =============================================================================
# FIGURE:
# PRIMARY NEGATIVE-CROSS-ENTROPY ANCHORED PREDICTIVE SYNERGY
#
# Panel A:
#   Standalone gain = B01 - B00
#   Conditional gain = B11 - B10
#
# Panel B:
#   S1 = (B11 - B10) - (B01 - B00)
#   with 95% bootstrap CI and FDR q
#
# Main manuscript figure
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------------------------------------------------
# PATHS
# -------------------------------------------------------------------------

BASE = str(RESULT_DIR)

PRIMARY_FILE = os.path.join(
    BASE,
    "TABLE_primary_S1_negative_CE.csv"
)

# We also need the subject-aggregated configuration metrics
METRIC_FILE = os.path.join(
    BASE,
    "CE_VS_PRAUC_CALIBRATION_DIAGNOSTICS",
    "FINAL_subject_aggregated_S1_by_metric.csv"
)

OUT_FILE = os.path.join(
    BASE,
    "FIGURE_PRIMARY_negative_CE_anchored_predictive_synergy.png"
)

OUT_FILE_PDF = os.path.join(
    BASE,
    "FIGURE_PRIMARY_negative_CE_anchored_predictive_synergy.pdf"
)


# -------------------------------------------------------------------------
# LOAD PRIMARY S1 RESULTS
# -------------------------------------------------------------------------

primary = pd.read_csv(
    PRIMARY_FILE
)

metric_df = pd.read_csv(
    METRIC_FILE
)

metric_df = metric_df[
    metric_df["Metric"] == "Negative_CE"
].copy()


# -------------------------------------------------------------------------
# STANDARDIZE EXTENSION LABELS
# -------------------------------------------------------------------------

label_map = {
    "atopy": "Atopy / exposures",
    "drug": "Drugomics",
    "lipid": "Lipidomics",
    "metab": "Metabolomics",
    "protein": "Proteomics"
}

primary["Extension_key"] = (
    primary["Extension"]
    .astype(str)
    .str.lower()
    .str.strip()
)

metric_df["Extension_key"] = (
    metric_df["Extension"]
    .astype(str)
    .str.lower()
    .str.strip()
)

primary["Label"] = (
    primary["Extension_key"]
    .map(label_map)
)

metric_df["Label"] = (
    metric_df["Extension_key"]
    .map(label_map)
)


# -------------------------------------------------------------------------
# ORDER
# -------------------------------------------------------------------------

order = [
    "atopy",
    "drug",
    "metab",
    "protein",
    "lipid"
]

primary["order"] = (
    primary["Extension_key"]
    .map({
        k: i
        for i, k in enumerate(order)
    })
)

metric_df["order"] = (
    metric_df["Extension_key"]
    .map({
        k: i
        for i, k in enumerate(order)
    })
)

primary = (
    primary
    .sort_values("order")
    .reset_index(drop=True)
)

metric_df = (
    metric_df
    .sort_values("order")
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------
# MERGE
# -------------------------------------------------------------------------

plot_df = primary.merge(
    metric_df[
        [
            "Extension_key",
            "Standalone_gain",
            "Conditional_gain"
        ]
    ],
    on="Extension_key",
    how="left"
)


# -------------------------------------------------------------------------
# Y POSITIONS
# -------------------------------------------------------------------------

y = np.arange(
    len(plot_df)
)


# =============================================================================
# FIGURE
# =============================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 7),
    gridspec_kw={
        "width_ratios": [1.2, 1]
    }
)

ax1, ax2 = axes


# =============================================================================
# PANEL A — PREDICTIVE CONTRIBUTION
# =============================================================================

for i, row in plot_df.iterrows():

    # horizontal connector
    ax1.plot(
        [
            row["Standalone_gain"],
            row["Conditional_gain"]
        ],
        [i, i],
        linewidth=2
    )

    # standalone
    ax1.scatter(
        row["Standalone_gain"],
        i,
        s=90,
        zorder=3,
        label=(
            "Extension alone: B01 − B00"
            if i == 0
            else None
        )
    )

    # conditional
    ax1.scatter(
        row["Conditional_gain"],
        i,
        marker="s",
        s=90,
        zorder=3,
        label=(
            "Added to anchor: B11 − B10"
            if i == 0
            else None
        )
    )


ax1.axvline(
    0,
    linestyle="--",
    linewidth=1.5
)

ax1.set_yticks(y)
ax1.set_yticklabels(
    plot_df["Label"]
)

ax1.invert_yaxis()

ax1.set_xlabel(
    "Change in negative cross-entropy\n(higher = better)"
)

ax1.set_title(
    "A   Predictive contribution",
    loc="left",
    fontweight="bold"
)

ax1.legend(
    frameon=False,
    loc="lower right"
)

ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)


# =============================================================================
# PANEL B — PRIMARY S1
# =============================================================================

x = plot_df["S1_negative_CE"].values

lower = (
    x
    - plot_df["CI_lower"].values
)

upper = (
    plot_df["CI_upper"].values
    - x
)

ax2.errorbar(
    x,
    y,
    xerr=[
        lower,
        upper
    ],
    fmt="o",
    markersize=8,
    capsize=4,
    linewidth=2
)

ax2.axvline(
    0,
    linestyle="--",
    linewidth=1.5
)

ax2.set_yticks(y)
ax2.set_yticklabels(
    plot_df["Label"]
)

ax2.invert_yaxis()

ax2.set_xlabel(
    r"$S_1$ (negative cross-entropy)"
)

ax2.set_title(
    "B   Anchor–extension interaction",
    loc="left",
    fontweight="bold"
)

ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)


# -------------------------------------------------------------------------
# Q-VALUE LABELS
# -------------------------------------------------------------------------

xmin, xmax = ax2.get_xlim()

text_x = xmax + 0.03 * (
    xmax - xmin
)

for i, row in plot_df.iterrows():

    q = row["FDR_q"]

    if q < 0.001:
        qtxt = "q<0.001"
    else:
        qtxt = f"q={q:.3f}"

    ax2.text(
        text_x,
        i,
        qtxt,
        va="center",
        fontsize=10
    )


# extend x-axis to make room for labels
ax2.set_xlim(
    xmin,
    text_x + 0.12 * (
        xmax - xmin
    )
)


# -------------------------------------------------------------------------
# ANNOTATE DIRECTION
# -------------------------------------------------------------------------

ax2.text(
    0.02,
    -0.12,
    "Sub-additive",
    transform=ax2.transAxes,
    ha="left",
    fontsize=10
)

ax2.text(
    0.98,
    -0.12,
    "Super-additive",
    transform=ax2.transAxes,
    ha="right",
    fontsize=10
)


# =============================================================================
# GLOBAL TITLE
# =============================================================================

fig.suptitle(
    "Primary anchored predictive synergy analysis in U-BIOPRED",
    fontsize=17,
    y=0.98
)

plt.tight_layout(
    rect=[
        0,
        0.03,
        1,
        0.94
    ]
)


# =============================================================================
# SAVE
# =============================================================================

plt.savefig(
    OUT_FILE,
    dpi=400,
    bbox_inches="tight"
)

plt.savefig(
    OUT_FILE_PDF,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(OUT_FILE)
print(OUT_FILE_PDF)

In [ ]:
# =============================================================================
# FIGURE:
# STANDALONE PREDICTIVE VALUE DOES NOT IMPLY INCREMENTAL VALUE
#
# Primary metric: Negative cross-entropy
#
# x-axis = Standalone gain   = B01 - B00
# y-axis = Conditional gain  = B11 - B10
#
# S1 = Conditional gain - Standalone gain
#
# Above diagonal: S1 > 0  (super-additive direction)
# Below diagonal: S1 < 0  (sub-additive direction)
# =============================================================================

import os
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# 1. PATHS
# =============================================================================

BASE = str(RESULT_DIR)

INPUT_FILE = os.path.join(
    BASE,
    "CE_VS_PRAUC_CALIBRATION_DIAGNOSTICS",
    "FINAL_subject_aggregated_S1_by_metric.csv"
)

OUT_PNG = os.path.join(
    BASE,
    "FIGURE_negative_CE_standalone_vs_conditional_gain.png"
)

OUT_PDF = os.path.join(
    BASE,
    "FIGURE_negative_CE_standalone_vs_conditional_gain.pdf"
)


# =============================================================================
# 2. LOAD SUBJECT-AGGREGATED RESULTS
# =============================================================================

df = pd.read_csv(
    INPUT_FILE
)

df = df[
    df["Metric"] == "Negative_CE"
].copy()


# =============================================================================
# 3. LABELS
# =============================================================================

label_map = {
    "atopy": "Atopy / exposures",
    "drug": "Drugomics",
    "lipid": "Lipidomics",
    "metab": "Metabolomics",
    "protein": "Proteomics"
}

df["Label"] = (
    df["Extension"]
    .astype(str)
    .str.lower()
    .map(label_map)
)


# =============================================================================
# 4. CHECK VALUES
# =============================================================================

print("=" * 90)
print("NEGATIVE CE: STANDALONE VS CONDITIONAL GAIN")
print("=" * 90)

display(
    df[
        [
            "Extension",
            "Standalone_gain",
            "Conditional_gain",
            "S1"
        ]
    ].round(4)
)


# =============================================================================
# 5. CREATE FIGURE
# =============================================================================

fig, ax = plt.subplots(
    figsize=(9, 8)
)


# -----------------------------------------------------------------------------
# determine plotting limits
# -----------------------------------------------------------------------------

all_values = pd.concat(
    [
        df["Standalone_gain"],
        df["Conditional_gain"]
    ]
)

margin = (
    all_values.max()
    - all_values.min()
) * 0.20

xmin = all_values.min() - margin
xmax = all_values.max() + margin

ymin = xmin
ymax = xmax


# -----------------------------------------------------------------------------
# zero reference lines
# -----------------------------------------------------------------------------

ax.axhline(
    0,
    linewidth=1
)

ax.axvline(
    0,
    linewidth=1
)


# -----------------------------------------------------------------------------
# S1 = 0 diagonal
# -----------------------------------------------------------------------------

ax.plot(
    [xmin, xmax],
    [xmin, xmax],
    linestyle="--",
    linewidth=1.5
)


# -----------------------------------------------------------------------------
# points
# -----------------------------------------------------------------------------

ax.scatter(
    df["Standalone_gain"],
    df["Conditional_gain"],
    s=100,
    zorder=3
)


# -----------------------------------------------------------------------------
# labels
# -----------------------------------------------------------------------------

for _, row in df.iterrows():

    ax.annotate(
        row["Label"],
        (
            row["Standalone_gain"],
            row["Conditional_gain"]
        ),
        xytext=(7, 6),
        textcoords="offset points",
        fontsize=11
    )


# =============================================================================
# 6. AXES AND ANNOTATIONS
# =============================================================================

ax.set_xlim(
    xmin,
    xmax
)

ax.set_ylim(
    ymin,
    ymax
)

ax.set_xlabel(
    r"Standalone gain: $B_{01}-B_{00}$ (negative cross-entropy)",
    fontsize=12
)

ax.set_ylabel(
    r"Conditional gain: $B_{11}-B_{10}$ (negative cross-entropy)",
    fontsize=12
)

ax.set_title(
    "Standalone predictive value does not imply incremental value",
    fontsize=16,
    fontweight="bold"
)


# -----------------------------------------------------------------------------
# interpretation labels
# -----------------------------------------------------------------------------

ax.text(
    0.04,
    0.94,
    "Above diagonal: positive $S_1$",
    transform=ax.transAxes,
    fontsize=11
)

ax.text(
    0.04,
    0.89,
    "Below diagonal: negative $S_1$",
    transform=ax.transAxes,
    fontsize=11
)


# -----------------------------------------------------------------------------
# clean styling
# -----------------------------------------------------------------------------

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()


# =============================================================================
# 7. SAVE
# =============================================================================

plt.savefig(
    OUT_PNG,
    dpi=400,
    bbox_inches="tight"
)

plt.savefig(
    OUT_PDF,
    bbox_inches="tight"
)

plt.show()


print("\nSaved:")
print(OUT_PNG)
print(OUT_PDF)

In [ ]:
# ======================================================================================
# SUPPLEMENTARY FIGURE:
# SUBJECT-AGGREGATED OOF PROBABILITY DISTRIBUTION AND CALIBRATION
#
# Fully reproducible from saved OOF prediction files.
# ======================================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve


# ======================================================================================
# 1. PATHS
# ======================================================================================

BASE = str(RESULT_DIR)

OUT_DIR = os.path.join(
    BASE,
    "CE_VS_PRAUC_CALIBRATION_DIAGNOSTICS"
)

os.makedirs(
    OUT_DIR,
    exist_ok=True
)


# ======================================================================================
# 2. FILE MAP
# ======================================================================================

FILES = {
    "B10": os.path.join(
        BASE,
        "B10_anchor_OOF_predictions.csv"
    ),

    "atopy": os.path.join(
        BASE,
        "B11_atopy_OOF_predictions.csv"
    ),

    "drug": os.path.join(
        BASE,
        "B11_drug_OOF_predictions.csv"
    ),

    "lipid": os.path.join(
        BASE,
        "B11_lipid_OOF_predictions.csv"
    ),

    "metab": os.path.join(
        BASE,
        "B11_metab_OOF_predictions.csv"
    ),

    "protein": os.path.join(
        BASE,
        "B11_protein_OOF_predictions.csv"
    )
}


# ======================================================================================
# 3. LOADER
# ======================================================================================

def load_subject_aggregated(path):

    x = pd.read_csv(path)

    required = [
        "subject",
        "y_true",
        "p1"
    ]

    missing = [
        c for c in required
        if c not in x.columns
    ]

    if missing:
        raise ValueError(
            f"{os.path.basename(path)} missing columns: {missing}"
        )

    x["p1"] = pd.to_numeric(
        x["p1"],
        errors="coerce"
    )

    x["y_true"] = pd.to_numeric(
        x["y_true"],
        errors="coerce"
    )

    x = x.dropna(
        subset=[
            "subject",
            "y_true",
            "p1"
        ]
    )

    # ----------------------------------------------------------
    # PRIMARY ANALYSIS AGGREGATION:
    # average repeated OOF probability within subject
    # ----------------------------------------------------------

    s = (
        x.groupby(
            "subject",
            as_index=False
        )
        .agg(
            y_true=("y_true", "first"),
            probability=("p1", "mean"),
            N_predictions=("p1", "size")
        )
    )

    return s


# ======================================================================================
# 4. LOAD ALL MODELS
# ======================================================================================

anchor = load_subject_aggregated(
    FILES["B10"]
)

models = {
    "Anchor": anchor,

    "Anchor + atopy":
        load_subject_aggregated(
            FILES["atopy"]
        ),

    "Anchor + drugomics":
        load_subject_aggregated(
            FILES["drug"]
        ),

    "Anchor + lipidomics":
        load_subject_aggregated(
            FILES["lipid"]
        ),

    "Anchor + metabolomics":
        load_subject_aggregated(
            FILES["metab"]
        ),

    "Anchor + proteomics":
        load_subject_aggregated(
            FILES["protein"]
        )
}


# ======================================================================================
# 5. SANITY CHECKS
# ======================================================================================

print("=" * 100)
print("SUBJECT-AGGREGATED OOF CALIBRATION DATA")
print("=" * 100)

for name, d in models.items():

    print(
        f"{name:24s} "
        f"N={len(d):3d}  "
        f"mean repeated predictions/subject="
        f"{d['N_predictions'].mean():.1f}"
    )

    if len(d) != 215:
        raise ValueError(
            f"{name}: expected 215 subjects, found {len(d)}"
        )

print("\nAnchor outcome counts:")
print(
    anchor["y_true"]
    .value_counts()
    .sort_index()
)


# ======================================================================================
# 6. FIGURE
# ======================================================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

ax1, ax2 = axes


# ======================================================================================
# PANEL A — ANCHOR PROBABILITY DISTRIBUTION
# ======================================================================================

negative = anchor[
    anchor["y_true"] == 0
]

positive = anchor[
    anchor["y_true"] == 1
]

bins = np.linspace(
    0,
    1,
    11
)

ax1.hist(
    negative["probability"],
    bins=bins,
    density=True,
    alpha=0.55,
    label="Observed <2 exacerbations"
)

ax1.hist(
    positive["probability"],
    bins=bins,
    density=True,
    alpha=0.55,
    label="Observed ≥2 exacerbations"
)

ax1.set_xlim(
    0,
    1
)

ax1.set_xlabel(
    "Subject-aggregated predicted probability"
)

ax1.set_ylabel(
    "Density"
)

ax1.set_title(
    "A   Anchor probability distribution",
    loc="left",
    fontweight="bold"
)

ax1.legend(
    frameon=False,
    fontsize=9
)


# ======================================================================================
# PANEL B — CALIBRATION CURVES
# ======================================================================================

ax2.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.3,
    label="Perfect calibration"
)

for name, d in models.items():

    prob_true, prob_pred = calibration_curve(
        d["y_true"],
        d["probability"],
        n_bins=8,
        strategy="quantile"
    )

    ax2.plot(
        prob_pred,
        prob_true,
        marker="o",
        linewidth=1.4,
        markersize=5,
        label=name
    )

ax2.set_xlim(
    0,
    1
)

ax2.set_ylim(
    0,
    1
)

ax2.set_xlabel(
    "Mean predicted probability"
)

ax2.set_ylabel(
    "Observed event frequency"
)

ax2.set_title(
    "B   Subject-aggregated calibration",
    loc="left",
    fontweight="bold"
)

ax2.legend(
    frameon=False,
    fontsize=8,
    loc="upper left"
)


# ======================================================================================
# 7. CLEAN FORMATTING
# ======================================================================================

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()


# ======================================================================================
# 8. SAVE
# ======================================================================================

PNG = os.path.join(
    OUT_DIR,
    "SUPPLEMENTARY_subject_aggregated_probability_calibration.png"
)

PDF = os.path.join(
    OUT_DIR,
    "SUPPLEMENTARY_subject_aggregated_probability_calibration.pdf"
)

plt.savefig(
    PNG,
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    PDF,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(PNG)
print(PDF)

In [ ]:
# =============================================================================
# ANALYSIS:
# ANALYSIS_DRUGOMICS_CLUSTER1_VS_CLUSTER0_DELTA_S1_BOOTSTRAP5000
#
# PURPOSE:
# Compare negative-cross-entropy S1 directly between Drugomics Cluster 1
# and Cluster 0.
#
# S1 = (B11 - B10) - (B01 - B00)
#
# Primary performance functional:
# Negative cross-entropy (higher = better)
#
# Bootstrap:
# Resample subjects independently within Cluster 0 and Cluster 1.
#
# Outputs:
#   1. Cluster-specific observed S1 and bootstrap CI
#   2. Delta S1 = S1_Cluster1 - S1_Cluster0
#   3. Bootstrap 95% CI and two-sided p-value for Delta S1
#   4. Bootstrap boxplot
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# 1. PATHS
# =============================================================================

BASE = (
    "/Users/nz1413/IntegrAO/results/ubiopred/"
    "FINAL_S1_30x5/subgroup_S1"
)

PROFILE_FILE = os.path.join(
    BASE,
    "subject_complementarity_profiles.csv"
)

CLUSTER_FILE = os.path.join(
    BASE,
    "drugomics_feature_dissection",
    "drugomics_subject_level_analysis_dataset.csv"
)

OUT_DIR = os.path.join(
    BASE,
    "drugomics_feature_dissection",
    "DELTA_S1_CLUSTER1_VS_CLUSTER0"
)

os.makedirs(
    OUT_DIR,
    exist_ok=True
)


# =============================================================================
# 2. LOAD
# =============================================================================

profiles = pd.read_csv(
    PROFILE_FILE
)

clusters = pd.read_csv(
    CLUSTER_FILE
)

print("=" * 100)
print(
    "ANALYSIS_DRUGOMICS_CLUSTER1_VS_CLUSTER0_"
    "DELTA_S1_BOOTSTRAP5000"
)
print("=" * 100)

print("\nProfiles shape:", profiles.shape)
print("Cluster dataset shape:", clusters.shape)

print("\nProfile columns:")
print(list(profiles.columns))

print("\nCluster dataset columns:")
print(list(clusters.columns))


# =============================================================================
# 3. RESTRICT TO DRUGOMICS
# =============================================================================

if "extension" in profiles.columns:

    profiles["extension"] = (
        profiles["extension"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    drug = profiles[
        profiles["extension"].isin(
            ["drug", "drugomics"]
        )
    ].copy()

else:

    drug = profiles.copy()


print("\nDrugomics profile N:", len(drug))


# =============================================================================
# 4. IDENTIFY SUBJECT ID
# =============================================================================

def find_id_col(df):

    candidates = [
        "subject",
        "patient",
        "subject_id",
        "patient_id",
        "ID",
        "id"
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise ValueError(
        "Could not identify subject ID column."
    )


PROFILE_ID = find_id_col(
    drug
)

CLUSTER_ID = find_id_col(
    clusters
)

print("\nProfile ID:", PROFILE_ID)
print("Cluster ID:", CLUSTER_ID)


drug[PROFILE_ID] = (
    drug[PROFILE_ID]
    .astype(str)
    .str.strip()
)

clusters[CLUSTER_ID] = (
    clusters[CLUSTER_ID]
    .astype(str)
    .str.strip()
)


# =============================================================================
# 5. CHECK REQUIRED PREDICTION COLUMNS
# =============================================================================

required_predictions = [
    "p00",
    "p01",
    "p10",
    "p11"
]

missing = [
    c for c in required_predictions
    if c not in drug.columns
]

if missing:
    raise ValueError(
        f"Missing prediction columns: {missing}"
    )


# =============================================================================
# 6. FIND OUTCOME COLUMN
# =============================================================================

outcome_candidates = [
    "y_true",
    "outcome",
    "Outcome",
    "y",
    "Y",
    "label",
    "target"
]

OUTCOME = None

for c in outcome_candidates:

    if c in drug.columns:
        OUTCOME = c
        break


if OUTCOME is None:

    # try to obtain it from cluster dataset
    for c in outcome_candidates:

        if c in clusters.columns:
            OUTCOME = c

            clusters = clusters[
                [
                    CLUSTER_ID,
                    OUTCOME,
                    "Cluster1"
                ]
            ].copy()

            drug = drug.merge(
                clusters,
                left_on=PROFILE_ID,
                right_on=CLUSTER_ID,
                how="inner"
            )

            break


if OUTCOME is None:
    raise ValueError(
        "Could not identify outcome column. "
        "Please inspect the printed columns and tell me its name."
    )


print("\nOutcome column:", OUTCOME)


# =============================================================================
# 7. ADD CLUSTER MEMBERSHIP IF NEEDED
# =============================================================================

if "Cluster1" not in drug.columns:

    if "Cluster1" not in clusters.columns:
        raise ValueError(
            "Cluster1 column not found."
        )

    cluster_small = (
        clusters[
            [
                CLUSTER_ID,
                "Cluster1"
            ]
        ]
        .drop_duplicates()
        .copy()
    )

    drug = drug.merge(
        cluster_small,
        left_on=PROFILE_ID,
        right_on=CLUSTER_ID,
        how="inner"
    )


drug["Cluster1"] = pd.to_numeric(
    drug["Cluster1"],
    errors="coerce"
)

drug[OUTCOME] = pd.to_numeric(
    drug[OUTCOME],
    errors="coerce"
)

for c in required_predictions:

    drug[c] = pd.to_numeric(
        drug[c],
        errors="coerce"
    )


drug = drug.dropna(
    subset=[
        "Cluster1",
        OUTCOME,
        "p00",
        "p01",
        "p10",
        "p11"
    ]
).copy()

drug = drug[
    drug["Cluster1"].isin(
        [0, 1]
    )
].copy()


# =============================================================================
# 8. HARD SANITY CHECK
# =============================================================================

print("\nCluster counts:")
print(
    drug["Cluster1"]
    .value_counts()
    .sort_index()
)

n0 = int(
    (drug["Cluster1"] == 0)
    .sum()
)

n1 = int(
    (drug["Cluster1"] == 1)
    .sum()
)

print("\nCluster 0 N =", n0)
print("Cluster 1 N =", n1)
print("Total N =", len(drug))


if n0 != 89 or n1 != 126:

    raise ValueError(
        f"Expected Cluster0=89 and Cluster1=126; "
        f"found {n0} and {n1}."
    )

print("\n✓ Correct drugomics population confirmed: 89 vs 126")


# =============================================================================
# 9. NEGATIVE CROSS-ENTROPY
# =============================================================================

EPS = 1e-12


def negative_ce(y, p):

    y = np.asarray(
        y,
        dtype=float
    )

    p = np.asarray(
        p,
        dtype=float
    )

    p = np.clip(
        p,
        EPS,
        1 - EPS
    )

    ce = -(
        y * np.log(p)
        +
        (1 - y) * np.log(1 - p)
    )

    return -np.mean(
        ce
    )


# =============================================================================
# 10. S1 FUNCTION
# =============================================================================

def calculate_s1(df):

    y = df[
        OUTCOME
    ].values

    B00 = negative_ce(
        y,
        df["p00"].values
    )

    B01 = negative_ce(
        y,
        df["p01"].values
    )

    B10 = negative_ce(
        y,
        df["p10"].values
    )

    B11 = negative_ce(
        y,
        df["p11"].values
    )

    standalone = (
        B01 - B00
    )

    conditional = (
        B11 - B10
    )

    S1 = (
        conditional
        - standalone
    )

    return {
        "B00": B00,
        "B01": B01,
        "B10": B10,
        "B11": B11,
        "Standalone_gain":
            standalone,
        "Conditional_gain":
            conditional,
        "S1":
            S1
    }


# =============================================================================
# 11. OBSERVED S1
# =============================================================================

cluster0 = drug[
    drug["Cluster1"] == 0
].copy()

cluster1 = drug[
    drug["Cluster1"] == 1
].copy()


obs0 = calculate_s1(
    cluster0
)

obs1 = calculate_s1(
    cluster1
)

delta_obs = (
    obs1["S1"]
    -
    obs0["S1"]
)


print("\n")
print("=" * 100)
print("OBSERVED DRUGOMICS CLUSTER S1")
print("=" * 100)

print(
    f"\nCluster 0 S1 = "
    f"{obs0['S1']:.4f}"
)

print(
    f"Cluster 1 S1 = "
    f"{obs1['S1']:.4f}"
)

print(
    f"\nDelta S1 "
    f"(Cluster 1 - Cluster 0) = "
    f"{delta_obs:.4f}"
)


# =============================================================================
# 12. BOOTSTRAP
# =============================================================================

N_BOOT = 5000

rng = np.random.default_rng(
    123
)

boot0 = np.empty(
    N_BOOT
)

boot1 = np.empty(
    N_BOOT
)

boot_delta = np.empty(
    N_BOOT
)


for b in range(
    N_BOOT
):

    # ---------------------------------------------------------
    # independently resample subjects within each cluster
    # ---------------------------------------------------------

    idx0 = rng.integers(
        0,
        len(cluster0),
        len(cluster0)
    )

    idx1 = rng.integers(
        0,
        len(cluster1),
        len(cluster1)
    )

    d0 = (
        cluster0
        .iloc[idx0]
    )

    d1 = (
        cluster1
        .iloc[idx1]
    )

    s0 = (
        calculate_s1(
            d0
        )["S1"]
    )

    s1 = (
        calculate_s1(
            d1
        )["S1"]
    )

    boot0[b] = s0
    boot1[b] = s1

    boot_delta[b] = (
        s1 - s0
    )



In [ ]:
# =============================================================================
# SUPPLEMENTARY FIGURE S1
# PCA LOADINGS DEFINING THE THREE DRUGOMICS AXES
#
# Uses SAVED loading files only.
# No need for the original "analysis" object.
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# PATHS
# =============================================================================

BASE = (
    "/Users/nz1413/IntegrAO/results/ubiopred/"
    "FINAL_S1_30x5/subgroup_S1/drugomics_feature_dissection"
)

ANDROGEN_FILE = os.path.join(
    BASE,
    "androgen_axis_loadings.csv"
)

CORTICO_FILE = os.path.join(
    BASE,
    "corticosteroid_axis_loadings.csv"
)

BRONCHO_FILE = os.path.join(
    BASE,
    "bronchodilator_axis_loadings.csv"
)

OUT_PNG = os.path.join(
    BASE,
    "SUPPLEMENTARY_FIGURE_S1_drugomics_PCA_axis_loadings.png"
)

OUT_PDF = os.path.join(
    BASE,
    "SUPPLEMENTARY_FIGURE_S1_drugomics_PCA_axis_loadings.pdf"
)


# =============================================================================
# LOAD
# =============================================================================

androgen = pd.read_csv(ANDROGEN_FILE)
cortico = pd.read_csv(CORTICO_FILE)
broncho = pd.read_csv(BRONCHO_FILE)


# =============================================================================
# CLEAN LABELS
# =============================================================================

def clean_name(x):

    x = str(x)

    x = x.replace("..ng.mL.", "")
    x = x.replace(".ng.mL.", "")
    x = x.replace("..DHEA.s.", " (DHEA-S)")

    x = x.replace(".", " ")

    x = " ".join(
        x.split()
    )

    return x


for df in [
    androgen,
    cortico,
    broncho
]:

    df["Label"] = (
        df["Feature"]
        .apply(clean_name)
    )


# =============================================================================
# SORT BY LOADING
# =============================================================================

# For androgen, showing all 17 is possible but visually dense.
# Show the 10 strongest absolute loadings in the figure.
androgen["Abs"] = (
    androgen["Loading"].abs()
)

androgen_plot = (
    androgen
    .sort_values(
        "Abs",
        ascending=False
    )
    .head(10)
    .sort_values(
        "Loading"
    )
)

cortico_plot = (
    cortico
    .sort_values(
        "Loading"
    )
)

broncho_plot = (
    broncho
    .sort_values(
        "Loading"
    )
)


# =============================================================================
# FIGURE
# =============================================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 7)
)

axA, axB, axC = axes


# =============================================================================
# A — ANDROGEN
# =============================================================================

y = np.arange(
    len(androgen_plot)
)

axA.barh(
    y,
    androgen_plot["Loading"]
)

axA.axvline(
    0,
    linestyle="--",
    linewidth=1
)

axA.set_yticks(y)

axA.set_yticklabels(
    androgen_plot["Label"],
    fontsize=8
)

axA.set_xlabel(
    "PC1 loading"
)

axA.set_title(
    "A   Androgen / endogenous steroid",
    loc="left",
    fontweight="bold"
)


# =============================================================================
# B — CORTICOSTEROID
# =============================================================================

y = np.arange(
    len(cortico_plot)
)

axB.barh(
    y,
    cortico_plot["Loading"]
)

axB.axvline(
    0,
    linestyle="--",
    linewidth=1
)

axB.set_yticks(y)

axB.set_yticklabels(
    cortico_plot["Label"],
    fontsize=9
)

axB.set_xlabel(
    "PC1 loading"
)

axB.set_title(
    "B   Corticosteroid-related",
    loc="left",
    fontweight="bold"
)


# =============================================================================
# C — BRONCHODILATOR
# =============================================================================

y = np.arange(
    len(broncho_plot)
)

axC.barh(
    y,
    broncho_plot["Loading"]
)

axC.axvline(
    0,
    linestyle="--",
    linewidth=1
)

axC.set_yticks(y)

axC.set_yticklabels(
    broncho_plot["Label"],
    fontsize=9
)

axC.set_xlabel(
    "PC1 loading"
)

axC.set_title(
    "C   Bronchodilator",
    loc="left",
    fontweight="bold"
)


# =============================================================================
# FORMAT
# =============================================================================

for ax in axes:

    ax.spines[
        "top"
    ].set_visible(False)

    ax.spines[
        "right"
    ].set_visible(False)


fig.suptitle(
    "PCA loadings defining the exploratory drugomics summary axes",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout(
    rect=[
        0,
        0,
        1,
        0.94
    ]
)


# =============================================================================
# SAVE
# =============================================================================

plt.savefig(
    OUT_PNG,
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    OUT_PDF,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(OUT_PNG)
print(OUT_PDF)

In [ ]:
# =============================================================================
# SUPPLEMENTARY FIGURE S2
# PCA-DERIVED DRUGOMICS AXES AND CLUSTER 1 MEMBERSHIP
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------------

BASE = (
    "/Users/nz1413/IntegrAO/results/ubiopred/"
    "FINAL_S1_30x5/subgroup_S1/drugomics_feature_dissection"
)

FILE = os.path.join(
    BASE,
    "drugomics_latent_axes_bootstrap.csv"
)

OUT_PNG = os.path.join(
    BASE,
    "SUPPLEMENTARY_FIGURE_S2_drugomics_PC1_cluster1.png"
)

OUT_PDF = os.path.join(
    BASE,
    "SUPPLEMENTARY_FIGURE_S2_drugomics_PC1_cluster1.pdf"
)

# -----------------------------------------------------------------------------
# READ RESULTS
# -----------------------------------------------------------------------------

df = pd.read_csv(FILE)

wanted = [
    "Androgen_axis",
    "Corticosteroid_axis",
    "Bronchodilator_axis"
]

plot_df = (
    df[df["Predictor"].isin(wanted)]
    .copy()
    .set_index("Predictor")
    .loc[wanted]
    .reset_index()
)

labels = {
    "Androgen_axis":
        "Androgen / endogenous steroid PC1",

    "Corticosteroid_axis":
        "Corticosteroid-related PC1",

    "Bronchodilator_axis":
        "Bronchodilator PC1"
}

plot_df["Label"] = plot_df["Predictor"].map(labels)

# -----------------------------------------------------------------------------
# PLOT
# -----------------------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8.2, 4.8)
)

y = np.arange(len(plot_df))

x = plot_df["OR_per_1SD"].values
lo = plot_df["CI_lower"].values
hi = plot_df["CI_upper"].values

ax.errorbar(
    x,
    y,
    xerr=np.vstack([
        x - lo,
        hi - x
    ]),
    fmt="o",
    markersize=8,
    capsize=4,
    linewidth=1.6
)

# Null OR
ax.axvline(
    1,
    linestyle="--",
    linewidth=1.2
)

ax.set_yticks(y)
ax.set_yticklabels(
    plot_df["Label"],
    fontsize=11
)

ax.invert_yaxis()

ax.set_xlabel(
    "Adjusted odds ratio for Cluster 1 per 1-SD increase in PC1",
    fontsize=11
)

ax.set_title(
    "PCA-derived drugomics profiles associated with Cluster 1",
    fontsize=14,
    fontweight="bold",
    pad=14
)

# -----------------------------------------------------------------------------
# ADD RESULTS TO RIGHT OF EACH CI
# -----------------------------------------------------------------------------

xmax = max(hi) * 1.45

ax.set_xlim(
    max(0.25, min(lo) * 0.80),
    xmax
)

for i, row in plot_df.iterrows():

    txt = (
        f"OR {row['OR_per_1SD']:.2f} "
        f"({row['CI_lower']:.2f}–{row['CI_upper']:.2f})"
        f"\n"
        f"p={row['p_value']:.3f}; "
        f"q={row['FDR_q']:.3f}"
    )

    ax.text(
        row["CI_upper"] + 0.05,
        i,
        txt,
        va="center",
        fontsize=9
    )

# -----------------------------------------------------------------------------
# CLEAN FORMAT
# -----------------------------------------------------------------------------

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.text(
    0.01,
    -0.20,
    "Odds ratios are adjusted for the other PCA axes, sex, age, BMI and smoking status.",
    transform=ax.transAxes,
    fontsize=9
)

plt.tight_layout()

# -----------------------------------------------------------------------------
# SAVE
# -----------------------------------------------------------------------------

plt.savefig(
    OUT_PNG,
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    OUT_PDF,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(OUT_PNG)
print(OUT_PDF)

In [ ]:
# =============================================================================
# SUPPLEMENTARY FIGURE S3
# FULL CLINICAL CHARACTERIZATION OF DRUGOMICS CLUSTER 1 VS CLUSTER 0
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# 1. PATH
# =============================================================================

BASE = (
    "/Users/nz1413/IntegrAO/results/ubiopred/"
    "FINAL_S1_30x5/subgroup_S1/drugomics_feature_dissection/"
    "FINAL_CLUSTER1_VS_CLUSTER0_126_VS_89_FULL_CLINICAL"
)

FILE = os.path.join(
    BASE,
    "TABLE_continuous_full_clinical_cluster1_vs_cluster0.csv"
)

OUT_PNG = os.path.join(
    BASE,
    "SUPPLEMENTARY_FIGURE_S3_full_clinical_characterization.png"
)

OUT_PDF = os.path.join(
    BASE,
    "SUPPLEMENTARY_FIGURE_S3_full_clinical_characterization.pdf"
)


# =============================================================================
# 2. LOAD
# =============================================================================

df = pd.read_csv(FILE)

print("Shape:", df.shape)

if len(df) != 19:
    raise ValueError(
        f"Expected 19 clinical variables, found {len(df)}"
    )


# =============================================================================
# 3. CLEAN DISPLAY LABELS
# =============================================================================

label_map = {

    "AsthmaBurgenSCore_weighted":
        "Weighted asthma burden",

    "AsthmaBurgenSCore":
        "Asthma burden",

    "Exacerbation_Per_Year":
        "Prior-year exacerbations",

    "FEV1_Change":
        "Bronchodilator FEV1 change",

    "ACQ5_Total_Raw":
        "ACQ5",

    "AQLQ_Average":
        "AQLQ",

    "MD_per_year":
        "Medical encounters / year",

    "hosp_LOS":
        "Hospital length of stay",

    "MARS_Total_Raw":
        "MARS adherence score",

    "sumED_per_year":
        "Emergency visits / year",

    "sputum_Neutrophils":
        "Sputum neutrophils",

    "Oral_Corticosteroids_Normalised_Dose_.mg.":
        "Maintenance OCS dose",

    "ICU_LOS":
        "ICU length of stay",

    "Body_Mass_Index_kgm2":
        "BMI",

    "FEV1_Pre_Salbutamol":
        "Pre-BD FEV1",

    "Age":
        "Age",

    "FVC_Pre_Salbutamol":
        "Pre-BD FVC",

    "FEV1_Post_Salbutamol":
        "Post-BD FEV1",

    "FEV1FVC_Post_Salbutamol_Actual_Ratio":
        "Post-BD FEV1/FVC"
}

df["Label"] = (
    df["Variable"]
    .map(label_map)
    .fillna(df["Variable"])
)


# =============================================================================
# 4. NUMERIC CONVERSION
# =============================================================================

for col in [
    "Rank_biserial",
    "p_value",
    "FDR_q"
]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# =============================================================================
# 5. ORDER BY EFFECT SIZE
# largest negative effects at top
# =============================================================================

df = (
    df
    .sort_values(
        "Rank_biserial",
        ascending=True
    )
    .reset_index(drop=True)
)


# =============================================================================
# 6. FIGURE
# =============================================================================

fig, ax = plt.subplots(
    figsize=(10, 9.5)
)

y = np.arange(
    len(df)
)


# =============================================================================
# 7. CONNECT POINTS TO ZERO
# =============================================================================

for i, row in df.iterrows():

    ax.plot(
        [
            0,
            row["Rank_biserial"]
        ],
        [
            i,
            i
        ],
        linewidth=1,
        zorder=1
    )


# =============================================================================
# 8. NON-SIGNIFICANT RESULTS
# =============================================================================

nonsig = (
    df["FDR_q"] >= 0.05
)

ax.scatter(
    df.loc[
        nonsig,
        "Rank_biserial"
    ],
    y[nonsig],
    s=55,
    facecolors="white",
    edgecolors="black",
    linewidth=1.1,
    zorder=3
)


# =============================================================================
# 9. FDR-SIGNIFICANT RESULTS
# =============================================================================

sig = (
    df["FDR_q"] < 0.05
)

ax.scatter(
    df.loc[
        sig,
        "Rank_biserial"
    ],
    y[sig],
    s=70,
    zorder=4
)


# =============================================================================
# 10. ZERO REFERENCE
# =============================================================================

ax.axvline(
    0,
    linestyle="--",
    linewidth=1.2
)


# =============================================================================
# 11. LABELS
# =============================================================================

ax.set_yticks(
    y
)

ax.set_yticklabels(
    df["Label"],
    fontsize=9.5
)

ax.set_xlabel(
    "Rank-biserial effect size\n"
    "← Lower in Cluster 1                         Higher in Cluster 1 →",
    fontsize=11
)

ax.set_title(
    "Clinical characterization of Drugomics Cluster 1",
    fontsize=15,
    fontweight="bold",
    pad=14
)


# =============================================================================
# 12. ADD FDR q VALUES TO SIGNIFICANT RESULTS
# =============================================================================

for i, row in df.iterrows():

    if row["FDR_q"] < 0.05:

        q = row["FDR_q"]

        if q < 0.001:
            qtxt = "q<0.001"
        else:
            qtxt = f"q={q:.3f}"

        if row["Rank_biserial"] < 0:

            xpos = (
                row["Rank_biserial"]
                - 0.018
            )

            ha = "right"

        else:

            xpos = (
                row["Rank_biserial"]
                + 0.018
            )

            ha = "left"

        ax.text(
            xpos,
            i,
            qtxt,
            va="center",
            ha=ha,
            fontsize=8
        )


# =============================================================================
# 13. FORMAT
# =============================================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xlim(
    -0.48,
    0.32
)

ax.margins(
    y=0.03
)

ax.text(
    0.01,
    -0.08,
    (
        "Filled points: FDR q < 0.05; "
        "open points: FDR q ≥ 0.05."
    ),
    transform=ax.transAxes,
    fontsize=9
)

plt.tight_layout()


# =============================================================================
# 14. SAVE
# =============================================================================

plt.savefig(
    OUT_PNG,
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    OUT_PDF,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(OUT_PNG)
print(OUT_PDF)

In [ ]:
# =============================================================================
# PUBLICATION-READY MAIN FIGURE 3
#
# Prediction-profile heterogeneity and exploratory characterization
# of the drugomics extension
#
# A-C  : Methods
# D-F  : Heterogeneity results
# G-I  : Exploratory characterization
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


# =============================================================================
# 1. PATHS
# =============================================================================

BASE = (
    "/Users/nz1413/IntegrAO/results/ubiopred/"
    "FINAL_S1_30x5/subgroup_S1/drugomics_feature_dissection"
)

SUBJECT_FILE = os.path.join(
    BASE,
    "drugomics_subject_level_analysis_dataset.csv"
)

DELTA_DIR = os.path.join(
    BASE,
    "DELTA_S1_CLUSTER1_VS_CLUSTER0"
)

S1_TABLE_FILE = os.path.join(
    DELTA_DIR,
    "TABLE_drugomics_cluster_negativeCE_delta_S1_bootstrap5000.csv"
)

S1_BOOT_FILE = os.path.join(
    DELTA_DIR,
    "BOOTSTRAP_drugomics_cluster_negativeCE_S1_5000.csv"
)

CLINICAL_FILE = os.path.join(
    BASE,
    "FINAL_CLUSTER1_VS_CLUSTER0_126_VS_89_FULL_CLINICAL",
    "TABLE_continuous_full_clinical_cluster1_vs_cluster0.csv"
)

AXIS_FILE = os.path.join(
    BASE,
    "drugomics_latent_axes_bootstrap.csv"
)

FINAL_DIR = os.path.join(
    BASE,
    "FINAL"
)

LOW_PRED_MODEL_FILE = os.path.join(
    FINAL_DIR,
    "FINAL_low_prednisolone_adjusted_model.csv"
)

LOW_PRED_SUMMARY_FILE = os.path.join(
    FINAL_DIR,
    "FINAL_low_prednisolone_group_summary.csv"
)

OUT_PNG = os.path.join(
    BASE,
    "MAIN_FIGURE_3_PUBLICATION_READY.png"
)

OUT_PDF = os.path.join(
    BASE,
    "MAIN_FIGURE_3_PUBLICATION_READY.pdf"
)


# =============================================================================
# 2. GLOBAL STYLE
# =============================================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.linewidth": 0.8
})

TITLE_FS = 11
LABEL_FS = 9
TICK_FS = 8
SMALL_FS = 7.5


# =============================================================================
# 3. LOAD DATA
# =============================================================================

subjects = pd.read_csv(SUBJECT_FILE)
s1_table = pd.read_csv(S1_TABLE_FILE)
s1_boot = pd.read_csv(S1_BOOT_FILE)
clinical = pd.read_csv(CLINICAL_FILE)
axis_df = pd.read_csv(AXIS_FILE)
low_pred_model = pd.read_csv(LOW_PRED_MODEL_FILE)
low_pred_summary = pd.read_csv(LOW_PRED_SUMMARY_FILE)


# =============================================================================
# 4. CHECK AND PREPARE SUBJECT DATA
# =============================================================================

required = [
    "patient",
    "Cluster1",
    "p00",
    "p01",
    "p10",
    "p11",
    "G_standalone",
    "G_conditional"
]

missing = [
    c for c in required
    if c not in subjects.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

for c in [
    "Cluster1",
    "p00",
    "p01",
    "p10",
    "p11",
    "G_standalone",
    "G_conditional"
]:
    subjects[c] = pd.to_numeric(
        subjects[c],
        errors="coerce"
    )

subjects = subjects.dropna(
    subset=required
).copy()

subjects["Cluster1"] = (
    subjects["Cluster1"]
    .astype(int)
)

assert len(subjects) == 215
assert (subjects["Cluster1"] == 0).sum() == 89
assert (subjects["Cluster1"] == 1).sum() == 126


# =============================================================================
# 5. USE SAVED SUBJECT-LEVEL GAIN VARIABLES
# =============================================================================

subjects["G_standalone_saved"] = (
    subjects["G_standalone"]
)

subjects["G_conditional_saved"] = (
    subjects["G_conditional"]
)

subjects["C_subject"] = (
    subjects["G_conditional_saved"]
    -
    subjects["G_standalone_saved"]
)


# =============================================================================
# 6. SIX-DIMENSIONAL PREDICTION PROFILE
# =============================================================================

features = [
    "p00",
    "p01",
    "p10",
    "p11",
    "G_standalone_saved",
    "G_conditional_saved"
]

feature_labels = [
    r"$p_{00}$",
    r"$p_{01}$",
    r"$p_{10}$",
    r"$p_{11}$",
    r"$G_{\rm stand}$",
    r"$G_{\rm cond}$"
]

X = subjects[
    features
].copy()

scaler = StandardScaler()
Xz = scaler.fit_transform(X)


# =============================================================================
# 7. HEATMAP DATA
# =============================================================================

heat = pd.DataFrame(
    Xz,
    columns=features
)

heat["Cluster1"] = (
    subjects["Cluster1"]
    .values
)

heat["C_subject"] = (
    subjects["C_subject"]
    .values
)

heat = (
    heat
    .sort_values(
        [
            "Cluster1",
            "C_subject"
        ],
        ascending=[
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

heat_matrix = (
    heat[
        features
    ]
    .values
)

n0 = int(
    (heat["Cluster1"] == 0).sum()
)

n1 = int(
    (heat["Cluster1"] == 1).sum()
)


# =============================================================================
# 8. PCA VISUALIZATION
# =============================================================================

pca = PCA(
    n_components=2
)

pcs = pca.fit_transform(
    Xz
)

subjects["Prediction_PC1"] = (
    pcs[:, 0]
)

subjects["Prediction_PC2"] = (
    pcs[:, 1]
)

pc1_var = (
    pca.explained_variance_ratio_[0]
    * 100
)

pc2_var = (
    pca.explained_variance_ratio_[1]
    * 100
)


# =============================================================================
# 9. S1 RESULTS
# =============================================================================

def find_result(text):

    rows = s1_table[
        s1_table["Comparison"]
        .astype(str)
        .str.contains(
            text,
            case=False,
            regex=False
        )
    ]

    if len(rows) != 1:
        raise ValueError(
            f"Could not identify row: {text}"
        )

    return rows.iloc[0]


r0 = find_result(
    "Cluster 0 S1"
)

r1 = find_result(
    "Cluster 1 S1"
)

rd = s1_table[
    s1_table["Comparison"]
    .astype(str)
    .str.contains(
        "Delta",
        case=False
    )
].iloc[0]


s0 = float(
    r0["Estimate"]
)

s1 = float(
    r1["Estimate"]
)

delta = float(
    rd["Estimate"]
)

delta_lo = float(
    rd["CI_lower"]
)

delta_hi = float(
    rd["CI_upper"]
)

delta_p = float(
    rd["p_value"]
)


boot0 = pd.to_numeric(
    s1_boot["Cluster0_S1"],
    errors="coerce"
).dropna().values

boot1 = pd.to_numeric(
    s1_boot["Cluster1_S1"],
    errors="coerce"
).dropna().values


# =============================================================================
# 10. CLINICAL CHARACTERIZATION
# =============================================================================

clinical_vars = [
    "AsthmaBurgenSCore_weighted",
    "Exacerbation_Per_Year",
    "ACQ5_Total_Raw",
    "AQLQ_Average"
]

clinical_label_map = {
    "AsthmaBurgenSCore_weighted":
        "Weighted asthma burden",
    "Exacerbation_Per_Year":
        "Prior-year exacerbations",
    "ACQ5_Total_Raw":
        "ACQ5",
    "AQLQ_Average":
        "AQLQ"
}

clinical_order = {
    x: i
    for i, x in enumerate(
        clinical_vars
    )
}

clin_plot = clinical[
    clinical["Variable"]
    .isin(
        clinical_vars
    )
].copy()

clin_plot["order"] = (
    clin_plot["Variable"]
    .map(
        clinical_order
    )
)

clin_plot["Label"] = (
    clin_plot["Variable"]
    .map(
        clinical_label_map
    )
)

clin_plot["Rank_biserial"] = pd.to_numeric(
    clin_plot["Rank_biserial"],
    errors="coerce"
)

clin_plot["FDR_q"] = pd.to_numeric(
    clin_plot["FDR_q"],
    errors="coerce"
)

clin_plot = (
    clin_plot
    .sort_values("order")
    .reset_index(drop=True)
)


# =============================================================================
# 11. DRUGOMICS LATENT AXES
# =============================================================================

axis_keep = [
    "Androgen_axis",
    "Corticosteroid_axis",
    "Bronchodilator_axis"
]

axis_label_map = {
    "Androgen_axis":
        "Androgen /\nendogenous steroid",
    "Corticosteroid_axis":
        "Corticosteroid-related",
    "Bronchodilator_axis":
        "Bronchodilator"
}

axis_order = {
    "Androgen_axis": 0,
    "Corticosteroid_axis": 1,
    "Bronchodilator_axis": 2
}

axis_plot = axis_df[
    axis_df["Predictor"]
    .isin(
        axis_keep
    )
].copy()

axis_plot["order"] = (
    axis_plot["Predictor"]
    .map(
        axis_order
    )
)

axis_plot["Label"] = (
    axis_plot["Predictor"]
    .map(
        axis_label_map
    )
)

axis_plot["OR"] = pd.to_numeric(
    axis_plot["OR_per_1SD"],
    errors="coerce"
)

axis_plot["CI_lower"] = pd.to_numeric(
    axis_plot["CI_lower"],
    errors="coerce"
)

axis_plot["CI_upper"] = pd.to_numeric(
    axis_plot["CI_upper"],
    errors="coerce"
)

axis_plot["FDR_q"] = pd.to_numeric(
    axis_plot["FDR_q"],
    errors="coerce"
)

axis_plot = (
    axis_plot
    .sort_values("order")
    .reset_index(drop=True)
)


# =============================================================================
# 12. LOW PREDNISOLONE RESULTS
# =============================================================================

n_low_c0 = 21
n_total_c0 = 53

n_low_c1 = 47
n_total_c1 = 64

pct_c0 = (
    100 * n_low_c0 / n_total_c0
)

pct_c1 = (
    100 * n_low_c1 / n_total_c1
)

pred_or = 4.89
pred_lo = 2.32
pred_hi = 10.83


# =============================================================================
# 13. FIGURE LAYOUT
# =============================================================================

fig = plt.figure(
    figsize=(
        14.8,
        12.0
    )
)

gs = fig.add_gridspec(
    3,
    3,
    height_ratios=[
        0.72,
        1.15,
        1.00
    ],
    width_ratios=[
        1.00,
        1.05,
        1.00
    ],
    left=0.08,
    right=0.98,
    bottom=0.07,
    top=0.93,
    hspace=0.43,
    wspace=0.42
)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[0, 2])

axD = fig.add_subplot(gs[1, 0])
axE = fig.add_subplot(gs[1, 1])
axF = fig.add_subplot(gs[1, 2])

axG = fig.add_subplot(gs[2, 0])
axH = fig.add_subplot(gs[2, 1])
axI = fig.add_subplot(gs[2, 2])


# =============================================================================
# 14. HELPER
# =============================================================================

def clean_method_axis(ax):

    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(False)


# =============================================================================
# A — FOUR PREDICTION STATES
# =============================================================================

clean_method_axis(
    axA
)

axA.set_title(
    "A   Four prediction states",
    loc="left",
    fontweight="bold",
    fontsize=TITLE_FS,
    pad=7
)

left = 0.23
bottom = 0.25
width = 0.55
height = 0.50

axA.add_patch(
    plt.Rectangle(
        (
            left,
            bottom
        ),
        width,
        height,
        fill=False,
        linewidth=1.1,
        transform=axA.transAxes
    )
)

axA.plot(
    [
        left + width/2,
        left + width/2
    ],
    [
        bottom,
        bottom + height
    ],
    transform=axA.transAxes,
    linewidth=0.9
)

axA.plot(
    [
        left,
        left + width
    ],
    [
        bottom + height/2,
        bottom + height/2
    ],
    transform=axA.transAxes,
    linewidth=0.9
)

cells = [
    (0.25, 0.75, r"$p_{00}$"),
    (0.75, 0.75, r"$p_{01}$"),
    (0.25, 0.25, r"$p_{10}$"),
    (0.75, 0.25, r"$p_{11}$")
]

for cx, cy, txt in cells:

    axA.text(
        left + width*cx,
        bottom + height*cy,
        txt,
        transform=axA.transAxes,
        ha="center",
        va="center",
        fontsize=15
    )

axA.text(
    left + width*0.25,
    bottom + height + 0.08,
    "Extension −",
    transform=axA.transAxes,
    ha="center",
    fontsize=8
)

axA.text(
    left + width*0.75,
    bottom + height + 0.08,
    "Extension +",
    transform=axA.transAxes,
    ha="center",
    fontsize=8
)

axA.text(
    left - 0.05,
    bottom + height*0.75,
    "Anchor −",
    transform=axA.transAxes,
    ha="right",
    va="center",
    fontsize=8
)

axA.text(
    left - 0.05,
    bottom + height*0.25,
    "Anchor +",
    transform=axA.transAxes,
    ha="right",
    va="center",
    fontsize=8
)


# =============================================================================
# B — SUBJECT-LEVEL PREDICTION DESCRIPTORS
# =============================================================================

clean_method_axis(
    axB
)

axB.set_title(
    "B   Subject-level prediction descriptors",
    loc="left",
    fontweight="bold",
    fontsize=TITLE_FS,
    pad=7
)

axB.text(
    0.50,
    0.72,
    r"$G_{\rm standalone,i}=p_{01,i}-p_{00,i}$",
    transform=axB.transAxes,
    ha="center",
    fontsize=12
)

axB.text(
    0.50,
    0.50,
    r"$G_{\rm conditional,i}=p_{11,i}-p_{10,i}$",
    transform=axB.transAxes,
    ha="center",
    fontsize=12
)

axB.text(
    0.50,
    0.28,
    r"$C_i=G_{\rm conditional,i}-G_{\rm standalone,i}$",
    transform=axB.transAxes,
    ha="center",
    fontsize=12
)

axB.text(
    0.50,
    0.07,
    (
        r"$C_i$ summarizes the subject-level prediction profile."
        "\n"
        r"Formal $S_1$ is evaluated at group level."
    ),
    transform=axB.transAxes,
    ha="center",
    fontsize=SMALL_FS
)


# =============================================================================
# C — EXPLORATORY CLUSTERING WORKFLOW
# =============================================================================

clean_method_axis(
    axC
)

axC.set_title(
    "C   Exploratory clustering workflow",
    loc="left",
    fontweight="bold",
    fontsize=TITLE_FS,
    pad=7
)


# Main workflow
steps = [
    "6-D prediction\nprofile",
    "Standardize",
    "K-means\nclustering",
    "Cluster-specific\n$S_1$"
]

xs = [
    0.12,
    0.38,
    0.64,
    0.89
]

box_widths = [
    0.19,
    0.17,
    0.17,
    0.19
]


for x, txt, bw in zip(
    xs,
    steps,
    box_widths
):

    axC.text(
        x,
        0.64,
        txt,
        transform=axC.transAxes,
        ha="center",
        va="center",
        fontsize=8.1,
        linespacing=1.05,
        bbox=dict(
            boxstyle="round,pad=0.34",
            facecolor="white",
            edgecolor="0.55",
            linewidth=0.9
        ),
        zorder=3
    )


for x1, x2 in zip(
    xs[:-1],
    xs[1:]
):

    axC.annotate(
        "",
        xy=(
            x2 - 0.085,
            0.64
        ),
        xytext=(
            x1 + 0.085,
            0.64
        ),
        xycoords=axC.transAxes,
        arrowprops=dict(
            arrowstyle="->",
            linewidth=1.0,
            shrinkA=0,
            shrinkB=0
        ),
        zorder=2
    )


# Silhouette comparison
axC.text(
    0.50,
    0.39,
    "Silhouette coefficient",
    transform=axC.transAxes,
    ha="center",
    va="center",
    fontsize=8.2,
    fontweight="bold"
)


sil_x = [
    0.30,
    0.50,
    0.70
]

sil_k = [
    "k = 2",
    "k = 3",
    "k = 4"
]

sil_v = [
    0.229,
    0.197,
    0.190
]


for x, klabel, val in zip(
    sil_x,
    sil_k,
    sil_v
):

    selected = (
        klabel
        == "k = 2"
    )

    axC.text(
        x,
        0.27,
        (
            f"{klabel}\n"
            f"{val:.3f}"
        ),
        transform=axC.transAxes,
        ha="center",
        va="center",
        fontsize=8.0,
        fontweight=(
            "bold"
            if selected
            else "normal"
        ),
        bbox=(
            dict(
                boxstyle="round,pad=0.24",
                facecolor="0.93",
                edgecolor="0.45",
                linewidth=0.9
            )
            if selected
            else None
        )
    )


axC.annotate(
    "selected",
    xy=(
        0.30,
        0.19
    ),
    xytext=(
        0.30,
        0.09
    ),
    xycoords=axC.transAxes,
    textcoords=axC.transAxes,
    ha="center",
    va="center",
    fontsize=7.5,
    fontweight="bold",
    arrowprops=dict(
        arrowstyle="-|>",
        linewidth=0.8
    )
)


# =============================================================================
# D — PREDICTION PROFILE HEATMAP
# =============================================================================

im = axD.imshow(
    heat_matrix,
    aspect="auto",
    interpolation="nearest",
    cmap="coolwarm",
    vmin=-2.5,
    vmax=2.5
)

axD.axhline(
    n0 - 0.5,
    linewidth=1.2
)

axD.set_xticks(
    np.arange(
        len(feature_labels)
    )
)

axD.set_xticklabels(
    feature_labels,
    rotation=40,
    ha="right",
    fontsize=TICK_FS
)

axD.set_yticks([])

axD.set_ylabel(
    "Participants",
    fontsize=LABEL_FS
)

axD.set_title(
    "D   Prediction profiles",
    loc="left",
    fontweight="bold",
    fontsize=TITLE_FS,
    pad=7
)

axD.text(
    -0.08,
    n0/2,
    "Cluster 0\nn=89",
    transform=axD.get_yaxis_transform(),
    ha="right",
    va="center",
    fontsize=8
)

axD.text(
    -0.08,
    n0 + n1/2,
    "Cluster 1\nn=126",
    transform=axD.get_yaxis_transform(),
    ha="right",
    va="center",
    fontsize=8
)

cbar = fig.colorbar(
    im,
    ax=axD,
    fraction=0.045,
    pad=0.025
)

cbar.set_label(
    "Standardized value",
    fontsize=7.5
)

cbar.ax.tick_params(
    labelsize=7
)


# =============================================================================
# E — PCA PROJECTION
# =============================================================================

c0 = subjects[
    subjects["Cluster1"]
    == 0
]

c1 = subjects[
    subjects["Cluster1"]
    == 1
]

axE.scatter(
    c0["Prediction_PC1"],
    c0["Prediction_PC2"],
    s=20,
    alpha=0.70,
    label="Cluster 0"
)

axE.scatter(
    c1["Prediction_PC1"],
    c1["Prediction_PC2"],
    s=20,
    alpha=0.70,
    label="Cluster 1"
)

for group in [
    c0,
    c1
]:

    axE.scatter(
        group["Prediction_PC1"].mean(),
        group["Prediction_PC2"].mean(),
        marker="X",
        s=90,
        edgecolor="black",
        linewidth=0.8,
        zorder=5
    )

axE.axhline(
    0,
    linewidth=0.6,
    alpha=0.5
)

axE.axvline(
    0,
    linewidth=0.6,
    alpha=0.5
)

axE.set_xlabel(
    f"PC1 ({pc1_var:.1f}% variance)",
    fontsize=LABEL_FS
)

axE.set_ylabel(
    f"PC2 ({pc2_var:.1f}% variance)",
    fontsize=LABEL_FS
)

axE.set_title(
    "E   PCA projection of prediction profiles",
    loc="left",
    fontweight="bold",
    fontsize=TITLE_FS,
    pad=7
)

axE.legend(
    frameon=False,
    loc="upper right",
    fontsize=7.5
)

axE.text(
    0.02,
    0.02,
    "PCA for visualization only",
    transform=axE.transAxes,
    fontsize=7,
    va="bottom"
)


# =============================================================================
# F — EXPLORATORY S1 DISTRIBUTIONS
# =============================================================================

bp = axF.boxplot(
    [
        boot0,
        boot1
    ],
    positions=[
        1,
        2
    ],
    widths=0.48,
    showfliers=False,
    patch_artist=True
)

for patch in bp["boxes"]:
    patch.set_alpha(
        0.12
    )

for median in bp["medians"]:
    median.set_linewidth(
        1.2
    )

rng = np.random.default_rng(
    2026
)

n_show = 100

show0 = rng.choice(
    boot0,
    size=n_show,
    replace=False
)

show1 = rng.choice(
    boot1,
    size=n_show,
    replace=False
)

x0 = (
    1
    +
    rng.normal(
        0,
        0.045,
        size=n_show
    )
)

x1 = (
    2
    +
    rng.normal(
        0,
        0.045,
        size=n_show
    )
)

axF.scatter(
    x0,
    show0,
    s=13,
    alpha=0.55,
    edgecolor="0.35",
    linewidth=0.25
)

axF.scatter(
    x1,
    show1,
    s=13,
    alpha=0.55,
    edgecolor="0.35",
    linewidth=0.25
)

axF.scatter(
    [
        1,
        2
    ],
    [
        s0,
        s1
    ],
    marker="D",
    s=65,
    edgecolor="black",
    linewidth=0.8,
    zorder=5,
    label="Observed $S_1$"
)

axF.axhline(
    0,
    linestyle="--",
    linewidth=0.9
)

annotation = (
    rf"$\Delta S_1={delta:.3f}$"
    "\n"
    rf"95% CI {delta_lo:.3f} to {delta_hi:.3f}"
    "\n"
    rf"$p={delta_p:.3f}$"
)

axF.text(
    0.50,
    0.97,
    annotation,
    transform=axF.transAxes,
    ha="center",
    va="top",
    fontsize=7.7,
    bbox=dict(
        boxstyle="round,pad=0.32",
        facecolor="white",
        edgecolor="0.7",
        linewidth=0.8
    )
)

axF.set_xticks(
    [
        1,
        2
    ]
)

axF.set_xticklabels(
    [
        "Cluster 0\n(n=89)",
        "Cluster 1\n(n=126)"
    ],
    fontsize=TICK_FS
)

axF.set_ylabel(
    r"$S_1$ (negative cross-entropy)",
    fontsize=LABEL_FS
)

axF.set_title(
    "F   Exploratory $S_1$ distributions",
    loc="left",
    fontweight="bold",
    fontsize=TITLE_FS,
    pad=7
)

axF.legend(
    frameon=False,
    loc="lower right",
    fontsize=7
)


# =============================================================================
# G — CLINICAL PHENOTYPE
# =============================================================================

yG = np.arange(
    len(
        clin_plot
    )
)

effectG = (
    clin_plot[
        "Rank_biserial"
    ]
    .values
)

axG.axvline(
    0,
    linestyle="--",
    linewidth=0.9
)

axG.hlines(
    yG,
    0,
    effectG,
    linewidth=1.0
)

axG.scatter(
    effectG,
    yG,
    s=45,
    zorder=3
)

axG.set_yticks(
    yG
)

axG.set_yticklabels(
    clin_plot[
        "Label"
    ],
    fontsize=8
)

axG.invert_yaxis()

axG.set_xlabel(
    (
        "Rank-biserial effect size\n"
        "← Lower in Cluster 1     Higher in Cluster 1 →"
    ),
    fontsize=8
)

axG.set_title(
    "G   Clinical phenotype",
    loc="left",
    fontweight="bold",
    fontsize=TITLE_FS,
    pad=7
)

for i, row in clin_plot.iterrows():

    q = float(
        row["FDR_q"]
    )

    qtxt = (
        "q<0.001"
        if q < 0.001
        else f"q={q:.3f}"
    )

    x = float(
        row["Rank_biserial"]
    )

    if x < 0:
        xpos = x - 0.025
        ha = "right"
    else:
        xpos = x + 0.025
        ha = "left"

    axG.text(
        xpos,
        i,
        qtxt,
        va="center",
        ha=ha,
        fontsize=7.2
    )

axG.set_xlim(
    -0.50,
    0.36
)


# =============================================================================
# H — DRUGOMICS PHENOTYPE
# =============================================================================

yH = np.arange(
    len(
        axis_plot
    )
)

orH = (
    axis_plot[
        "OR"
    ]
    .values
)

loH = (
    axis_plot[
        "CI_lower"
    ]
    .values
)

hiH = (
    axis_plot[
        "CI_upper"
    ]
    .values
)

axH.errorbar(
    orH,
    yH,
    xerr=np.vstack(
        [
            orH - loH,
            hiH - orH
        ]
    ),
    fmt="o",
    markersize=6,
    capsize=3,
    elinewidth=1.2,
    capthick=1.0,
    zorder=3
)

axH.axvline(
    1,
    linestyle="--",
    linewidth=0.9
)

axH.set_yticks(
    yH
)

axH.set_yticklabels(
    axis_plot[
        "Label"
    ],
    fontsize=8
)

axH.invert_yaxis()

axH.set_xlabel(
    "Adjusted OR per 1-SD increase in PC1",
    fontsize=8.5
)

axH.set_title(
    "H   Drugomics phenotype",
    loc="left",
    fontweight="bold",
    fontsize=TITLE_FS,
    pad=7
)

xmin = min(
    loH.min(),
    0.4
)

xmax = max(
    hiH.max(),
    2.2
)

axH.set_xlim(
    xmin - 0.08,
    xmax + 0.60
)

for i, row in axis_plot.iterrows():

    q = float(
        row["FDR_q"]
    )

    qtxt = (
        "q<0.001"
        if q < 0.001
        else f"q={q:.3f}"
    )

    txt = (
        f'{row["OR"]:.2f} '
        f'({row["CI_lower"]:.2f}–'
        f'{row["CI_upper"]:.2f}) '
        f'{qtxt}'
    )

    axH.text(
        xmax + 0.06,
        i,
        txt,
        va="center",
        ha="left",
        fontsize=7
    )


# =============================================================================
# I — LOW URINARY PREDNISOLONE
# =============================================================================

xI = np.arange(
    2
)

pct = [
    pct_c0,
    pct_c1
]

axI.bar(
    xI,
    pct,
    width=0.55
)

axI.set_xticks(
    xI
)

axI.set_xticklabels(
    [
        "Cluster 0\n(n=53 OCS users)",
        "Cluster 1\n(n=64 OCS users)"
    ],
    fontsize=8
)

axI.set_ylim(
    0,
    100
)

axI.set_ylabel(
    "Low urinary prednisolone (%)",
    fontsize=LABEL_FS
)

axI.set_title(
    "I   Low urinary prednisolone",
    loc="left",
    fontweight="bold",
    fontsize=TITLE_FS,
    pad=7
)

for i, (
    percentage,
    low_n,
    total_n
) in enumerate(
    [
        (
            pct_c0,
            n_low_c0,
            n_total_c0
        ),
        (
            pct_c1,
            n_low_c1,
            n_total_c1
        )
    ]
):

    axI.text(
        i,
        percentage + 3,
        (
            f"{percentage:.1f}%\n"
            f"({low_n}/{total_n})"
        ),
        ha="center",
        fontsize=7.8
    )

axI.text(
    0.50,
    0.94,
    (
        f"Adjusted OR {pred_or:.2f}\n"
        f"95% CI {pred_lo:.2f}–{pred_hi:.2f}\n"
        r"$q<0.001$"
    ),
    transform=axI.transAxes,
    ha="center",
    va="top",
    fontsize=7.6,
    bbox=dict(
        boxstyle="round,pad=0.30",
        facecolor="white",
        edgecolor="0.7",
        linewidth=0.8
    )
)


# =============================================================================
# 15. CLEAN RESULT PANELS
# =============================================================================

for ax in [
    axD,
    axE,
    axF,
    axG,
    axH,
    axI
]:

    ax.spines["top"].set_visible(
        False
    )

    ax.spines["right"].set_visible(
        False
    )

    ax.tick_params(
        width=0.8,
        length=3
    )


# =============================================================================
# 16. ROW LABELS
# =============================================================================

fig.text(
    0.025,
    0.82,
    "METHOD",
    rotation=90,
    fontsize=8.5,
    fontweight="bold",
    va="center"
)

fig.text(
    0.025,
    0.52,
    "HETEROGENEITY RESULT",
    rotation=90,
    fontsize=8.5,
    fontweight="bold",
    va="center"
)

fig.text(
    0.025,
    0.20,
    "EXPLORATORY CHARACTERIZATION",
    rotation=90,
    fontsize=8.5,
    fontweight="bold",
    va="center"
)


# =============================================================================
# 17. MAIN TITLE
# =============================================================================

fig.suptitle(
    (
        "Prediction-profile heterogeneity and exploratory "
        "characterization of the drugomics extension"
    ),
    fontsize=13.5,
    fontweight="bold",
    y=0.975
)


# =============================================================================
# 18. SAVE
# =============================================================================

plt.savefig(
    OUT_PNG,
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    OUT_PDF,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(OUT_PNG)
print(OUT_PDF)

## Release notes

- Superseded duplicate analysis blocks were removed rather than silently merged.
- Numerical results are still read from or generated into the same manuscript analysis outputs.
- The final 3×3 Figure 3 code is retained as the authoritative main heterogeneity figure.
- Raw cohort data, subject identifiers, and intermediate private data products should **not** be committed to a public repository.
- Before creating a public release, run the notebook from a fresh environment against the governed data and compare generated tables/figures with the submitted manuscript.
